# Bengali LLM Hallucination Detection — Full CV + Fusion Ensemble

This notebook is the cleaned and updated version of the proven hybrid pipeline.

## Main design

- **Context-present route:** lexical/context alignment classifier.
- **Null-context route:** BanglaBERT + QA retrieval + evidence verifier + Step-6 hybrid model.
- **Cross-validation:** five official folds with nested threshold/model selection.
- **Fusion:** route-wise soft voting and Logistic Regression stacking.
- **Safety:** the original hybrid prediction is retained whenever honest OOF fusion does not pass the acceptance rule.

## Required Kaggle input files

- `final_train_max.parquet`
- `official_folds.csv`
- `competition_test.csv`

Enable a T4/P100 GPU. Internet is required on the first run to download `csebuetnlp/banglabert`.


# Bengali Hallucination — Old 0.732 Null + New 0.8991 Context

This notebook starts with only the processed Hallucination dataset:

- `final_train_max.parquet`
- `official_folds.csv`
- `competition_test.csv`

Final routing:

- **Context-present rows:** new repo-inspired lexical/context model.
- **Null-context rows:** the old Step 2 → Step 3 → Step 4 verifier → Step 6 pipeline that produced the earlier 0.732 public-score system.
- The old Step 4 context model remains only as an internal evidence verifier for Step 6. It is not used for final context-row predictions.

Before the first complete run:

1. Select a Kaggle GPU accelerator.
2. Turn Internet **ON** so `csebuetnlp/banglabert` can be downloaded.
3. Attach only the processed Hallucination dataset.
4. Run all cells in order.


## 1. Environment, paths and processed-data discovery


In [ ]:
import os
import gc
import re
import json
import shutil
import random
import warnings
import difflib
import unicodedata
from pathlib import Path
from typing import Callable, Dict, Optional, Sequence

import joblib
import numpy as np
import pandas as pd
import torch

from IPython.display import display
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

try:
    from catboost import CatBoostClassifier
    HAS_CATBOOST = True
except Exception:
    HAS_CATBOOST = False

warnings.filterwarnings("ignore")

SEED = 42
INNER_SPLITS = 4
THRESHOLDS = np.arange(0.10, 0.901, 0.01)

INPUT_ROOT = Path("/kaggle/input")
WORKING_ROOT = Path("/kaggle/working")
PREPROCESSED_DIR = WORKING_ROOT / "bh_preprocessed"
REPO_CONTEXT_DIR = WORKING_ROOT / "repo_inspired_hybrid"
FINAL_DIR = WORKING_ROOT / "old_null_new_context_final"

RESET_PIPELINE = True

PIPELINE_DIRS = [
    WORKING_ROOT / "null_model_step2",
    WORKING_ROOT / "null_retrieval_step3",
    WORKING_ROOT / "context_model_step4",
    WORKING_ROOT / "null_step6_hybrid_verifier",
    REPO_CONTEXT_DIR,
    FINAL_DIR,
]

if RESET_PIPELINE:
    for directory in PIPELINE_DIRS:
        if directory.exists():
            shutil.rmtree(directory)

PREPROCESSED_DIR.mkdir(parents=True, exist_ok=True)
REPO_CONTEXT_DIR.mkdir(parents=True, exist_ok=True)
FINAL_DIR.mkdir(parents=True, exist_ok=True)

def seed_everything(seed=SEED):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

seed_everything()

def find_input_file(names):
    matches = []
    for name in names:
        matches.extend(path for path in INPUT_ROOT.rglob(name) if path.is_file())
    if not matches:
        raise FileNotFoundError(f"Could not find: {names}")
    preferred = [
        path for path in matches
        if "hallucination" in str(path).lower()
    ]
    pool = preferred or matches
    return sorted(set(pool), key=lambda path: (len(path.parts), str(path)))[0]

DATA_PATHS = {
    "final_train_max.parquet": find_input_file(["final_train_max.parquet"]),
    "official_folds.csv": find_input_file(["official_folds.csv"]),
    "competition_test.csv": find_input_file(["competition_test.csv"]),
}

for name, source in DATA_PATHS.items():
    destination = PREPROCESSED_DIR / name
    if destination.exists() or destination.is_symlink():
        destination.unlink()
    destination.symlink_to(source)

print("=" * 88)
print("DATA")
print("=" * 88)
for name, path in DATA_PATHS.items():
    print(name, "->", path)

print("\nPyTorch:", torch.__version__)
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError("Enable a Kaggle GPU accelerator before running.")

print("CatBoost available:", HAS_CATBOOST)
print("\nThe first run downloads csebuetnlp/banglabert.")
print("Turn Kaggle Internet ON for the first complete run.")


## 2. Train the new high-accuracy context branch


In [ ]:
NULL_STRINGS = {
    "", "nan", "none", "null", "[null]", "nil", "n/a",
    "na", "<na>", "[]", "{}",
}

BANGLA_TOKEN_PATTERN = re.compile(r"[\u0980-\u09FFa-zA-Z0-9]+")
NUMBER_PATTERN = re.compile(r"[০-৯0-9]+(?:[.,][০-৯0-9]+)?")
NEGATION_TERMS = (
    " না", " নয়", " নয়", " নেই", " never ", " not ", " no ",
)

def clean_text(value):
    if value is None:
        return ""
    try:
        if pd.isna(value):
            return ""
    except (TypeError, ValueError):
        pass
    text = unicodedata.normalize("NFC", str(value))
    text = (
        text.replace("\u200b", " ")
        .replace("\u200c", " ")
        .replace("\u200d", " ")
    )
    text = re.sub(r"\s+", " ", text).strip()
    return "" if text.lower() in NULL_STRINGS else text

def normalize_frame(frame, require_label):
    frame = frame.copy()
    if "prompt_bn" not in frame.columns and "question" in frame.columns:
        frame = frame.rename(columns={"question": "prompt_bn"})
    if "response_bn" not in frame.columns and "answer" in frame.columns:
        frame = frame.rename(columns={"answer": "response_bn"})
    for column in ["context", "prompt_bn", "response_bn"]:
        if column not in frame.columns:
            frame[column] = ""
        frame[column] = frame[column].map(clean_text)
    frame["route"] = np.where(frame["context"].eq(""), "null", "context")
    if require_label:
        frame["label"] = pd.to_numeric(frame["label"], errors="raise").astype(int)
    return frame

official = normalize_frame(
    pd.read_csv(PREPROCESSED_DIR / "official_folds.csv", keep_default_na=False),
    True,
)
competition_test = normalize_frame(
    pd.read_csv(PREPROCESSED_DIR / "competition_test.csv", keep_default_na=False),
    False,
)

official["fold"] = pd.to_numeric(official["fold"], errors="raise").astype(int)
official["official_position"] = np.arange(len(official), dtype=np.int64)
competition_test["test_position"] = np.arange(len(competition_test), dtype=np.int64)

context_train = official[official["route"].eq("context")].copy().reset_index(drop=True)
context_test = competition_test[competition_test["route"].eq("context")].copy().reset_index(drop=True)

def word_set(text):
    return set(BANGLA_TOKEN_PATTERN.findall(clean_text(text).lower()))

def jaccard_similarity(first, second):
    first_words = word_set(first)
    second_words = word_set(second)
    if not first_words or not second_words:
        return 0.0
    return len(first_words & second_words) / len(first_words | second_words)

def character_overlap(reference, candidate):
    reference_chars = set(clean_text(reference))
    candidate_chars = set(clean_text(candidate))
    if not candidate_chars:
        return 0.0
    return len(reference_chars & candidate_chars) / len(candidate_chars)

def lcs_ratio(first, second):
    return float(
        difflib.SequenceMatcher(
            None,
            clean_text(first),
            clean_text(second),
        ).ratio()
    )

def extract_numbers(text):
    return set(NUMBER_PATTERN.findall(clean_text(text)))

def number_mismatch(reference, candidate):
    reference_numbers = extract_numbers(reference)
    candidate_numbers = extract_numbers(candidate)
    if not candidate_numbers:
        return 0.0
    return float(not candidate_numbers.issubset(reference_numbers))

def has_negation(text):
    normalized = " " + clean_text(text).lower() + " "
    return any(term in normalized for term in NEGATION_TERMS)

def extract_context_lexical_features(frame):
    rows = []
    for row in frame.itertuples(index=False):
        context = clean_text(row.context)
        prompt = clean_text(row.prompt_bn)
        response = clean_text(row.response_bn)
        rows.append({
            "context_response_jaccard": jaccard_similarity(context, response),
            "context_response_char_overlap": character_overlap(context, response),
            "response_exact_in_context": float(
                bool(response) and response.lower() in context.lower()
            ),
            "context_response_lcs": lcs_ratio(context, response),
            "context_response_number_mismatch": number_mismatch(context, response),
            "context_response_negation_mismatch": float(
                has_negation(context) != has_negation(response)
            ),
            "prompt_response_jaccard": jaccard_similarity(prompt, response),
            "prompt_context_jaccard": jaccard_similarity(prompt, context),
            "context_length_scaled": min(len(context) / 5000.0, 5.0),
            "prompt_length_scaled": min(len(prompt) / 1000.0, 5.0),
            "response_length_scaled": min(len(response) / 1000.0, 5.0),
        })
    return pd.DataFrame(rows, index=frame.index)

def evaluate_binary(y_true, prediction):
    return {
        "accuracy": float(accuracy_score(y_true, prediction)),
        "f1_label0": float(
            f1_score(y_true, prediction, pos_label=0, zero_division=0)
        ),
        "f1_label1": float(
            f1_score(y_true, prediction, pos_label=1, zero_division=0)
        ),
        "macro_f1": float(
            f1_score(y_true, prediction, average="macro", zero_division=0)
        ),
        "confusion_matrix": confusion_matrix(
            y_true, prediction, labels=[0, 1]
        ).tolist(),
    }

def tune_threshold(y_true, probability_fair):
    best = None
    best_key = None
    for threshold in THRESHOLDS:
        prediction = (probability_fair >= threshold).astype(int)
        candidate = {
            "threshold": float(threshold),
            **evaluate_binary(y_true, prediction),
        }
        key = (
            candidate["f1_label0"],
            candidate["macro_f1"],
            candidate["accuracy"],
            candidate["f1_label1"],
        )
        if best is None or key > best_key:
            best = candidate
            best_key = key
    return best

def make_logistic_model(random_state):
    return Pipeline([
        ("scale", StandardScaler()),
        (
            "model",
            LogisticRegression(
                C=0.5,
                class_weight="balanced",
                solver="liblinear",
                max_iter=3000,
                random_state=random_state,
            ),
        ),
    ])

def make_catboost_model(random_state):
    return CatBoostClassifier(
        iterations=140,
        depth=2,
        learning_rate=0.03,
        l2_leaf_reg=18.0,
        random_strength=1.0,
        loss_function="Logloss",
        random_seed=random_state,
        verbose=False,
        allow_writing_files=False,
        thread_count=-1,
    )

def model_factories():
    factories = {"logistic_regression": make_logistic_model}
    if HAS_CATBOOST:
        factories["catboost"] = make_catboost_model
    return factories

def nested_oof_for_model(X, y, folds, model_factory, seed_offset):
    outer_probability = np.full(len(y), np.nan, dtype=np.float64)
    outer_prediction = np.full(len(y), -1, dtype=np.int64)
    fold_thresholds = []

    for outer_fold in sorted(np.unique(folds)):
        train_index = np.where(folds != outer_fold)[0]
        valid_index = np.where(folds == outer_fold)[0]
        X_outer_train = X[train_index]
        y_outer_train = y[train_index]

        inner_splits = min(
            INNER_SPLITS,
            int(np.bincount(y_outer_train).min()),
        )
        inner_cv = StratifiedKFold(
            n_splits=inner_splits,
            shuffle=True,
            random_state=SEED + seed_offset + int(outer_fold),
        )
        inner_probability = np.full(len(train_index), np.nan, dtype=np.float64)

        for inner_fold, (inner_train_local, inner_valid_local) in enumerate(
            inner_cv.split(X_outer_train, y_outer_train)
        ):
            model = model_factory(
                SEED + seed_offset + 100 + inner_fold + int(outer_fold) * 10
            )
            model.fit(
                X_outer_train[inner_train_local],
                y_outer_train[inner_train_local],
            )
            inner_probability[inner_valid_local] = model.predict_proba(
                X_outer_train[inner_valid_local]
            )[:, 1]

        threshold = float(
            tune_threshold(y_outer_train, inner_probability)["threshold"]
        )
        fold_thresholds.append(threshold)

        final_outer_model = model_factory(
            SEED + seed_offset + 1000 + int(outer_fold)
        )
        final_outer_model.fit(X_outer_train, y_outer_train)
        valid_probability = final_outer_model.predict_proba(X[valid_index])[:, 1]

        outer_probability[valid_index] = valid_probability
        outer_prediction[valid_index] = (
            valid_probability >= threshold
        ).astype(int)

    return {
        "probability": outer_probability,
        "prediction": outer_prediction,
        "fold_thresholds": fold_thresholds,
        "final_threshold": float(np.median(fold_thresholds)),
        "metrics": evaluate_binary(y, outer_prediction),
    }

def select_best_nested_model(X_frame, y, folds):
    X = X_frame.to_numpy(dtype=np.float32)
    results = {}
    factories = model_factories()

    for model_name, factory in factories.items():
        result = nested_oof_for_model(
            X=X,
            y=y,
            folds=folds,
            model_factory=factory,
            seed_offset=10_000,
        )
        results[model_name] = result
        print(model_name, json.dumps(result["metrics"], indent=2))

    best_name = max(
        results,
        key=lambda name: (
            results[name]["metrics"]["f1_label0"],
            results[name]["metrics"]["macro_f1"],
        ),
    )
    return best_name, results[best_name], factories[best_name]

def fit_fold_ensemble_and_predict(
    X_train,
    y_train,
    folds,
    X_test,
    model_factory,
):
    X_train_array = X_train.to_numpy(dtype=np.float32)
    X_test_array = X_test.to_numpy(dtype=np.float32)
    test_probabilities = []

    for fold in sorted(np.unique(folds)):
        train_index = np.where(folds != fold)[0]
        model = model_factory(SEED + 5000 + int(fold))
        model.fit(X_train_array[train_index], y_train[train_index])
        test_probabilities.append(model.predict_proba(X_test_array)[:, 1])
        joblib.dump(
            model,
            REPO_CONTEXT_DIR / f"context_meta_fold_{fold}.joblib",
        )

    return np.vstack(test_probabilities).mean(axis=0)

context_features_train = extract_context_lexical_features(context_train)
context_features_test = extract_context_lexical_features(context_test)

for column in ["nli_entailment", "nli_neutral", "nli_contradiction"]:
    context_features_train[column] = 0.5
    context_features_test[column] = 0.5

context_features_train = (
    context_features_train.replace([np.inf, -np.inf], np.nan).fillna(0.0)
)
context_features_test = (
    context_features_test.replace([np.inf, -np.inf], np.nan).fillna(0.0)
)

context_y = context_train["label"].to_numpy(dtype=np.int64)
context_folds = context_train["fold"].to_numpy(dtype=np.int64)

context_best_name, context_best_result, context_factory = (
    select_best_nested_model(
        context_features_train,
        context_y,
        context_folds,
    )
)

context_threshold = float(context_best_result["final_threshold"])
context_test_probability = fit_fold_ensemble_and_predict(
    context_features_train,
    context_y,
    context_folds,
    context_features_test,
    context_factory,
)
context_test_prediction = (
    context_test_probability >= context_threshold
).astype(int)

context_oof_output = context_train[
    ["id", "official_position", "fold", "label"]
].copy()
context_oof_output["context_probability_fair"] = (
    context_best_result["probability"]
)
context_oof_output["context_prediction"] = (
    context_best_result["prediction"]
)

context_test_output = context_test[
    ["id", "test_position"]
].copy()
context_test_output["context_probability_fair"] = context_test_probability
context_test_output["context_prediction"] = context_test_prediction

context_oof_output.to_csv(
    REPO_CONTEXT_DIR / "context_oof.csv",
    index=False,
)
context_test_output.to_csv(
    REPO_CONTEXT_DIR / "context_test.csv",
    index=False,
)

context_report = {
    "best_model": context_best_name,
    "threshold": context_threshold,
    "metrics": context_best_result["metrics"],
}
with open(
    REPO_CONTEXT_DIR / "context_metrics.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(context_report, file, ensure_ascii=False, indent=2)

print("=" * 88)
print("NEW CONTEXT MODEL")
print("=" * 88)
print(json.dumps(context_report, ensure_ascii=False, indent=2))


## 3. Rebuild the old null branch

The next four cells reproduce the old null pipeline:

1. Step 2: BanglaBERT null classifier.
2. Step 3: correct-answer retrieval ensemble.
3. Step 4: context specialist models used only by the verifier.
4. Step 6: hybrid evidence verifier and final null predictions.


### Step 2 — Null BanglaBERT


In [ ]:

"""
Bengali Hallucination Competition
STEP 2: Null-context model training with fold-safe external pretraining,
official-data adaptation, 5-fold OOF evaluation, threshold tuning,
and null-route test prediction.

Final label meaning:
    0 = hallucinated
    1 = fair / faithful

Expected preprocessing directory:
    /kaggle/working/bh_preprocessed

Expected files:
    final_train_max.parquet
    official_folds.csv
    competition_test.csv

Designed for Kaggle GPU (T4/P100). Uses a manual PyTorch loop to avoid
Transformers Trainer API-version incompatibilities.
"""

from __future__ import annotations

import gc
import json
import math
import os
import random
import re
import unicodedata
from contextlib import nullcontext
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Tuple

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from torch.nn.utils import clip_grad_norm_
from torch.optim import AdamW
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    get_linear_schedule_with_warmup,
)


# =============================================================================
# CONFIGURATION
# =============================================================================

@dataclass
class Config:
    input_dir: str = "/kaggle/working/bh_preprocessed"
    output_dir: str = "/kaggle/working/null_model_step2"

    # Bengali ELECTRA/BanglaBERT model.
    model_name: str = "csebuetnlp/banglabert"

    seed: int = 42
    n_folds: int = 5

    # Null route usually contains relatively short prompt-response pairs.
    max_length: int = 256

    # External pretraining: all balanced null-context external rows.
    external_epochs: int = 1
    external_batch_size: int = 16
    external_grad_accum: int = 2
    external_learning_rate: float = 2e-5
    external_weight_decay: float = 0.01
    external_warmup_ratio: float = 0.06

    # Fold-specific competition adaptation.
    # Official train rows are repeated; a balanced external subset is mixed in.
    official_repeat: int = 30
    adaptation_external_size: int = 5000
    adaptation_epochs: int = 4
    adaptation_batch_size: int = 16
    adaptation_grad_accum: int = 1
    adaptation_learning_rate: float = 8e-6
    adaptation_weight_decay: float = 0.01
    adaptation_warmup_ratio: float = 0.05

    eval_batch_size: int = 64
    num_workers: int = 2
    max_grad_norm: float = 1.0

    # The official metric is F1 for label 0.
    threshold_min: float = 0.05
    threshold_max: float = 0.95
    threshold_step: float = 0.005

    # Set to one fold, e.g. [0], for a quick debugging run.
    folds_to_run: Tuple[int, ...] = (0, 1, 2, 3, 4)

    # Set True only for a fast pipeline test.
    debug: bool = False
    debug_external_rows: int = 12000


CFG = Config()


# =============================================================================
# REPRODUCIBILITY AND TEXT CLEANING
# =============================================================================

NULL_STRINGS = {
    "", "nan", "none", "null", "nil", "n/a", "na", "[]", "{}",
    "[null]", '[""]', "()", "<na>",
}


def seed_everything(seed: int) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    # Deterministic behavior where possible.
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True


def clean_text(value: object) -> str:
    if value is None:
        return ""

    try:
        if pd.isna(value):
            return ""
    except (TypeError, ValueError):
        pass

    text = unicodedata.normalize("NFC", str(value))
    text = text.replace("\u200b", " ").replace("\u200c", " ").replace("\u200d", " ")
    text = re.sub(r"\s+", " ", text).strip()

    if text.lower() in NULL_STRINGS:
        return ""

    return text


def normalize_frame(
    df: pd.DataFrame,
    require_label: bool = True,
) -> pd.DataFrame:
    """
    Normalize train/validation/test frames.

    Training and official validation data require a binary label.
    Competition test data intentionally has no label, so call with
    require_label=False for that dataframe.
    """
    df = df.copy()

    required = ["context", "prompt_bn", "response_bn"]
    if require_label:
        required.append("label")

    missing = [column for column in required if column not in df.columns]

    if missing:
        raise ValueError(
            f"Missing columns: {missing}. Available columns: {df.columns.tolist()}"
        )

    for column in ["context", "prompt_bn", "response_bn"]:
        df[column] = df[column].map(clean_text)

    if "label" in df.columns:
        df["label"] = pd.to_numeric(
            df["label"],
            errors="raise",
        ).astype(int)

        if not set(df["label"].unique()).issubset({0, 1}):
            raise ValueError(
                f"Unexpected labels: {sorted(df['label'].unique())}"
            )

    if "route" not in df.columns:
        df["route"] = np.where(
            df["context"].eq(""),
            "null",
            "context",
        )
    else:
        df["route"] = (
            df["route"]
            .fillna("null")
            .astype(str)
            .str.strip()
            .str.lower()
            .replace({
                "nan": "null",
                "none": "null",
                "": "null",
            })
        )

    return df


def is_official_source(source: object) -> bool:
    text = clean_text(source).lower()
    return any(token in text for token in ["official", "competition", "provided_sample"])


# =============================================================================
# DATASET AND TOKENIZATION
# =============================================================================

class PairDataset(Dataset):
    def __init__(
        self,
        frame: pd.DataFrame,
        tokenizer,
        max_length: int,
        with_labels: bool = True,
    ) -> None:
        self.prompts = frame["prompt_bn"].astype(str).tolist()
        self.responses = frame["response_bn"].astype(str).tolist()
        self.with_labels = with_labels
        self.labels = frame["label"].astype(int).tolist() if with_labels else None
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self) -> int:
        return len(self.prompts)

    def __getitem__(self, index: int) -> Dict[str, object]:
        # Pair encoding gives the model an explicit separator between prompt and response.
        encoded = self.tokenizer(
            self.prompts[index],
            self.responses[index],
            truncation="longest_first",
            max_length=self.max_length,
            padding=False,
        )

        if self.with_labels:
            encoded["labels"] = self.labels[index]

        return encoded


def make_loader(
    frame: pd.DataFrame,
    tokenizer,
    max_length: int,
    batch_size: int,
    shuffle: bool,
    num_workers: int,
    with_labels: bool = True,
) -> DataLoader:
    dataset = PairDataset(
        frame=frame,
        tokenizer=tokenizer,
        max_length=max_length,
        with_labels=with_labels,
    )

    collator = DataCollatorWithPadding(
        tokenizer=tokenizer,
        padding="longest",
        return_tensors="pt",
    )

    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=num_workers,
        pin_memory=torch.cuda.is_available(),
        persistent_workers=(num_workers > 0),
        collate_fn=collator,
    )


# =============================================================================
# TRAINING AND INFERENCE
# =============================================================================

def autocast_context(device: torch.device):
    if device.type == "cuda":
        return torch.autocast(device_type="cuda", dtype=torch.float16)
    return nullcontext()


def train_model(
    model,
    train_loader: DataLoader,
    device: torch.device,
    epochs: int,
    learning_rate: float,
    weight_decay: float,
    warmup_ratio: float,
    grad_accum_steps: int,
    max_grad_norm: float,
    stage_name: str,
    validation_loader: Optional[DataLoader] = None,
) -> Tuple[object, Dict[str, float]]:
    optimizer = AdamW(
        model.parameters(),
        lr=learning_rate,
        weight_decay=weight_decay,
    )

    updates_per_epoch = math.ceil(len(train_loader) / grad_accum_steps)
    total_updates = max(1, updates_per_epoch * epochs)
    warmup_steps = int(total_updates * warmup_ratio)

    scheduler = get_linear_schedule_with_warmup(
        optimizer=optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_updates,
    )

    scaler = torch.amp.GradScaler(
        "cuda",
        enabled=(device.type == "cuda"),
    )

    best_state = None
    best_macro_f1 = -1.0
    best_metrics: Dict[str, float] = {}

    model.to(device)

    for epoch in range(1, epochs + 1):
        model.train()
        optimizer.zero_grad(set_to_none=True)

        running_loss = 0.0
        seen_batches = 0

        progress = tqdm(
            train_loader,
            desc=f"{stage_name} | epoch {epoch}/{epochs}",
            leave=True,
        )

        for step, batch in enumerate(progress, start=1):
            batch = {
                key: value.to(device, non_blocking=True)
                for key, value in batch.items()
            }

            with autocast_context(device):
                outputs = model(**batch)
                loss = outputs.loss / grad_accum_steps

            scaler.scale(loss).backward()

            should_update = (
                step % grad_accum_steps == 0
                or step == len(train_loader)
            )

            if should_update:
                scaler.unscale_(optimizer)
                clip_grad_norm_(model.parameters(), max_grad_norm)
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)

            running_loss += loss.item() * grad_accum_steps
            seen_batches += 1

            progress.set_postfix(
                loss=f"{running_loss / max(1, seen_batches):.4f}",
                lr=f"{scheduler.get_last_lr()[0]:.2e}",
            )

        epoch_metrics = {
            "epoch": float(epoch),
            "train_loss": running_loss / max(1, seen_batches),
        }

        if validation_loader is not None:
            val_prob, val_labels = predict_probabilities(
                model=model,
                loader=validation_loader,
                device=device,
                return_labels=True,
            )

            val_pred = (val_prob >= 0.5).astype(int)

            epoch_metrics.update(
                {
                    "val_accuracy": accuracy_score(val_labels, val_pred),
                    "val_f1_label0": f1_score(
                        val_labels,
                        val_pred,
                        pos_label=0,
                        zero_division=0,
                    ),
                    "val_f1_label1": f1_score(
                        val_labels,
                        val_pred,
                        pos_label=1,
                        zero_division=0,
                    ),
                    "val_macro_f1": f1_score(
                        val_labels,
                        val_pred,
                        average="macro",
                        zero_division=0,
                    ),
                }
            )

            print(json.dumps(epoch_metrics, indent=2))

            if epoch_metrics["val_macro_f1"] > best_macro_f1:
                best_macro_f1 = epoch_metrics["val_macro_f1"]
                best_metrics = epoch_metrics.copy()
                best_state = {
                    key: value.detach().cpu().clone()
                    for key, value in model.state_dict().items()
                }

        else:
            print(json.dumps(epoch_metrics, indent=2))

    if best_state is not None:
        model.load_state_dict(best_state)

    return model, best_metrics


@torch.no_grad()
def predict_probabilities(
    model,
    loader: DataLoader,
    device: torch.device,
    return_labels: bool,
) -> Tuple[np.ndarray, Optional[np.ndarray]]:
    model.eval()
    model.to(device)

    probabilities: List[np.ndarray] = []
    all_labels: List[np.ndarray] = []

    for batch in tqdm(loader, desc="Predicting", leave=False):
        labels = batch.pop("labels", None)

        batch = {
            key: value.to(device, non_blocking=True)
            for key, value in batch.items()
        }

        with autocast_context(device):
            logits = model(**batch).logits

        # Probability of class 1 = fair/faithful.
        prob_fair = torch.softmax(logits.float(), dim=-1)[:, 1]
        probabilities.append(prob_fair.cpu().numpy())

        if labels is not None:
            all_labels.append(labels.numpy())

    probs = np.concatenate(probabilities)

    if return_labels:
        labels_array = np.concatenate(all_labels)
        return probs, labels_array

    return probs, None


def evaluate_at_threshold(
    y_true: np.ndarray,
    prob_fair: np.ndarray,
    threshold: float,
) -> Dict[str, object]:
    y_pred = (prob_fair >= threshold).astype(int)

    return {
        "threshold": float(threshold),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "f1_label0": float(
            f1_score(y_true, y_pred, pos_label=0, zero_division=0)
        ),
        "f1_label1": float(
            f1_score(y_true, y_pred, pos_label=1, zero_division=0)
        ),
        "macro_f1": float(
            f1_score(y_true, y_pred, average="macro", zero_division=0)
        ),
        "confusion_matrix": confusion_matrix(
            y_true,
            y_pred,
            labels=[0, 1],
        ).tolist(),
        "classification_report": classification_report(
            y_true,
            y_pred,
            labels=[0, 1],
            target_names=["hallucinated_0", "fair_1"],
            zero_division=0,
            output_dict=True,
        ),
    }


def tune_threshold(
    y_true: np.ndarray,
    prob_fair: np.ndarray,
    minimum: float,
    maximum: float,
    step: float,
) -> Dict[str, object]:
    best: Optional[Dict[str, object]] = None

    thresholds = np.arange(minimum, maximum + 1e-9, step)

    for threshold in thresholds:
        metrics = evaluate_at_threshold(
            y_true=y_true,
            prob_fair=prob_fair,
            threshold=float(threshold),
        )

        if best is None:
            best = metrics
            continue

        # Primary competition objective: maximize Macro-F1.
        # Tie-break: label-0 F1, label-1 F1, then accuracy.
        current_key = (
            metrics["macro_f1"],
            metrics["f1_label0"],
            metrics["f1_label1"],
            metrics["accuracy"],
        )
        best_key = (
            best["macro_f1"],
            best["f1_label0"],
            best["f1_label1"],
            best["accuracy"],
        )

        if current_key > best_key:
            best = metrics

    assert best is not None
    return best


# =============================================================================
# DATA PREPARATION
# =============================================================================

def load_data(cfg: Config):
    input_dir = Path(cfg.input_dir)

    train_path = input_dir / "final_train_max.parquet"
    official_path = input_dir / "official_folds.csv"
    test_path = input_dir / "competition_test.csv"

    for path in [train_path, official_path, test_path]:
        if not path.exists():
            raise FileNotFoundError(f"Required file not found: {path}")

    final_train = normalize_frame(
        pd.read_parquet(train_path),
        require_label=True,
    )
    official = normalize_frame(
        pd.read_csv(official_path, keep_default_na=False),
        require_label=True,
    )
    test = normalize_frame(
        pd.read_csv(test_path, keep_default_na=False),
        require_label=False,
    )

    if "fold" not in official.columns:
        raise ValueError("official_folds.csv does not contain a 'fold' column.")

    official["fold"] = pd.to_numeric(
        official["fold"],
        errors="raise",
    ).astype(int)

    # External null data only. All official rows are taken from official_folds.csv
    # so held-out fold rows cannot leak into training.
    null_pool = final_train[final_train["route"].eq("null")].copy()

    if "source" in null_pool.columns:
        external_null = null_pool[
            ~null_pool["source"].map(is_official_source)
        ].copy()
    else:
        # Robust fallback: remove exact official triples.
        official_keys = set(
            zip(
                official["context"],
                official["prompt_bn"],
                official["response_bn"],
            )
        )
        mask = [
            (context, prompt, response) not in official_keys
            for context, prompt, response in zip(
                null_pool["context"],
                null_pool["prompt_bn"],
                null_pool["response_bn"],
            )
        ]
        external_null = null_pool[np.asarray(mask)].copy()

    official_null = official[official["route"].eq("null")].copy()
    test_null = test[test["route"].eq("null")].copy()

    if len(official_null) != 169:
        print(
            f"[WARN] Expected approximately 169 official null rows; "
            f"found {len(official_null)}."
        )

    if len(test_null) != 1155:
        print(
            f"[WARN] Expected approximately 1155 test null rows; "
            f"found {len(test_null)}."
        )

    return external_null, official_null, test, test_null


def balanced_sample(
    frame: pd.DataFrame,
    total_size: int,
    seed: int,
) -> pd.DataFrame:
    if total_size <= 0 or len(frame) <= total_size:
        return frame.sample(frac=1, random_state=seed).reset_index(drop=True)

    per_label = total_size // 2
    parts = []

    for label in [0, 1]:
        group = frame[frame["label"].eq(label)]
        n = min(per_label, len(group))
        parts.append(group.sample(n=n, random_state=seed + label))

    result = pd.concat(parts, ignore_index=True)

    remaining = total_size - len(result)

    if remaining > 0:
        used_indices = set(result.index.tolist())
        remainder_pool = frame.drop(index=list(used_indices), errors="ignore")
        if len(remainder_pool) > 0:
            extra = remainder_pool.sample(
                n=min(remaining, len(remainder_pool)),
                random_state=seed + 99,
            )
            result = pd.concat([result, extra], ignore_index=True)

    return result.sample(frac=1, random_state=seed).reset_index(drop=True)


def make_adaptation_frame(
    official_train: pd.DataFrame,
    external_null: pd.DataFrame,
    cfg: Config,
    fold: int,
) -> pd.DataFrame:
    official_repeated = pd.concat(
        [official_train] * cfg.official_repeat,
        ignore_index=True,
    )
    official_repeated["adaptation_origin"] = "official"

    external_subset = balanced_sample(
        frame=external_null,
        total_size=cfg.adaptation_external_size,
        seed=cfg.seed + fold * 101,
    )
    external_subset["adaptation_origin"] = "external"

    combined = pd.concat(
        [official_repeated, external_subset],
        ignore_index=True,
    )

    # Remove exact duplicates only across sources while preserving intentional
    # repetition of official rows for weighting. We therefore do not drop
    # duplicate official copies.
    combined = combined.sample(
        frac=1,
        random_state=cfg.seed + fold,
    ).reset_index(drop=True)

    return combined


# =============================================================================
# MAIN PIPELINE
# =============================================================================

def main() -> None:
    cfg = CFG
    seed_everything(cfg.seed)

    output_dir = Path(cfg.output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    with open(output_dir / "config.json", "w", encoding="utf-8") as file:
        json.dump(asdict(cfg), file, ensure_ascii=False, indent=2)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    print("=" * 88)
    print("STEP 2 — NULL-CONTEXT MODEL (v4 macro-F1 + index-fixed)")
    print("=" * 88)
    print("Device:", device)

    if device.type == "cuda":
        print("GPU:", torch.cuda.get_device_name(0))

    external_null, official_null, full_test, test_null = load_data(cfg)

    if cfg.debug:
        external_null = balanced_sample(
            external_null,
            total_size=min(cfg.debug_external_rows, len(external_null)),
            seed=cfg.seed,
        )

    print("\n[DATA]")
    print("External null rows:", len(external_null))
    print("Official null rows:", len(official_null))
    print("Test null rows:", len(test_null))
    print("\nExternal labels:")
    print(external_null["label"].value_counts())
    print("\nOfficial null fold/label:")
    print(pd.crosstab(official_null["fold"], official_null["label"], margins=True))

    tokenizer = AutoTokenizer.from_pretrained(
        cfg.model_name,
        use_fast=True,
    )

    # -------------------------------------------------------------------------
    # STAGE A: Train one shared external null-context base model.
    # -------------------------------------------------------------------------
    external_model_dir = output_dir / "external_null_base"

    # Reuse the already-trained external base from the earlier v2/v3 run.
    previous_external_base = Path(
        "/kaggle/working/null_model_step2/external_null_base"
    )
    if (
        not (external_model_dir / "config.json").exists()
        and (previous_external_base / "config.json").exists()
    ):
        import shutil
        shutil.copytree(
            previous_external_base,
            external_model_dir,
            dirs_exist_ok=True,
        )
        print(
            "\n[STAGE A] Reused existing external base from:",
            previous_external_base,
        )

    if (external_model_dir / "config.json").exists():
        print("\n[STAGE A] Existing external base found. Loading it.")
    else:
        print("\n[STAGE A] Training shared external null-context base model.")

        model = AutoModelForSequenceClassification.from_pretrained(
            cfg.model_name,
            num_labels=2,
            id2label={0: "HALLUCINATED", 1: "FAIR"},
            label2id={"HALLUCINATED": 0, "FAIR": 1},
        )

        external_loader = make_loader(
            frame=external_null,
            tokenizer=tokenizer,
            max_length=cfg.max_length,
            batch_size=cfg.external_batch_size,
            shuffle=True,
            num_workers=cfg.num_workers,
            with_labels=True,
        )

        model, _ = train_model(
            model=model,
            train_loader=external_loader,
            device=device,
            epochs=cfg.external_epochs,
            learning_rate=cfg.external_learning_rate,
            weight_decay=cfg.external_weight_decay,
            warmup_ratio=cfg.external_warmup_ratio,
            grad_accum_steps=cfg.external_grad_accum,
            max_grad_norm=cfg.max_grad_norm,
            stage_name="External null pretraining",
            validation_loader=None,
        )

        external_model_dir.mkdir(parents=True, exist_ok=True)
        model.save_pretrained(external_model_dir)
        tokenizer.save_pretrained(external_model_dir)

        del model, external_loader
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    # -------------------------------------------------------------------------
    # STAGE B: Fold-safe official adaptation and OOF/test prediction.
    # -------------------------------------------------------------------------
    oof_prob = np.full(len(official_null), np.nan, dtype=np.float32)
    oof_fold = np.full(len(official_null), -1, dtype=np.int32)

    test_fold_probabilities: List[np.ndarray] = []
    fold_metrics: List[Dict[str, object]] = []

    test_null_loader = make_loader(
        frame=test_null,
        tokenizer=tokenizer,
        max_length=cfg.max_length,
        batch_size=cfg.eval_batch_size,
        shuffle=False,
        num_workers=cfg.num_workers,
        with_labels=False,
    )

    # Create positions relative to the 169-row official-null subset.
    # Do not preserve the original 299-row dataframe index, because values such
    # as 199 are out of bounds for the 169-length OOF arrays.
    official_null = official_null.reset_index(drop=True)
    official_null["_official_null_index"] = np.arange(
        len(official_null),
        dtype=np.int64,
    )

    for fold in cfg.folds_to_run:
        print("\n" + "=" * 88)
        print(f"FOLD {fold}")
        print("=" * 88)

        fold_train = official_null[
            official_null["fold"].ne(fold)
        ].copy()
        fold_valid = official_null[
            official_null["fold"].eq(fold)
        ].copy()

        if fold_valid.empty:
            print(f"[WARN] Fold {fold} has no null-route validation rows. Skipping.")
            continue

        adaptation_frame = make_adaptation_frame(
            official_train=fold_train,
            external_null=external_null,
            cfg=cfg,
            fold=fold,
        )

        print("Official train rows:", len(fold_train))
        print("Official validation rows:", len(fold_valid))
        print("Adaptation rows:", len(adaptation_frame))
        print(pd.crosstab(
            adaptation_frame["adaptation_origin"],
            adaptation_frame["label"],
            margins=True,
        ))

        fold_model_dir = output_dir / f"fold_{fold}"

        model = AutoModelForSequenceClassification.from_pretrained(
            external_model_dir,
            num_labels=2,
        )

        train_loader = make_loader(
            frame=adaptation_frame,
            tokenizer=tokenizer,
            max_length=cfg.max_length,
            batch_size=cfg.adaptation_batch_size,
            shuffle=True,
            num_workers=cfg.num_workers,
            with_labels=True,
        )

        valid_loader = make_loader(
            frame=fold_valid,
            tokenizer=tokenizer,
            max_length=cfg.max_length,
            batch_size=cfg.eval_batch_size,
            shuffle=False,
            num_workers=cfg.num_workers,
            with_labels=True,
        )

        model, best_epoch_metrics = train_model(
            model=model,
            train_loader=train_loader,
            device=device,
            epochs=cfg.adaptation_epochs,
            learning_rate=cfg.adaptation_learning_rate,
            weight_decay=cfg.adaptation_weight_decay,
            warmup_ratio=cfg.adaptation_warmup_ratio,
            grad_accum_steps=cfg.adaptation_grad_accum,
            max_grad_norm=cfg.max_grad_norm,
            stage_name=f"Official adaptation fold {fold}",
            validation_loader=valid_loader,
        )

        fold_valid_prob, fold_valid_labels = predict_probabilities(
            model=model,
            loader=valid_loader,
            device=device,
            return_labels=True,
        )

        positions = fold_valid["_official_null_index"].to_numpy(
            dtype=np.int64
        )

        if positions.min() < 0 or positions.max() >= len(oof_prob):
            raise IndexError(
                "OOF position mapping is invalid: "
                f"min={positions.min()}, max={positions.max()}, "
                f"array_size={len(oof_prob)}"
            )

        oof_prob[positions] = fold_valid_prob
        oof_fold[positions] = fold

        fold_test_prob, _ = predict_probabilities(
            model=model,
            loader=test_null_loader,
            device=device,
            return_labels=False,
        )
        test_fold_probabilities.append(fold_test_prob)

        fold_default_metrics = evaluate_at_threshold(
            y_true=fold_valid_labels,
            prob_fair=fold_valid_prob,
            threshold=0.5,
        )
        fold_default_metrics["fold"] = int(fold)
        fold_default_metrics["best_training_epoch"] = best_epoch_metrics

        fold_metrics.append(fold_default_metrics)

        fold_model_dir.mkdir(parents=True, exist_ok=True)
        model.save_pretrained(fold_model_dir)
        tokenizer.save_pretrained(fold_model_dir)

        with open(
            fold_model_dir / "fold_metrics.json",
            "w",
            encoding="utf-8",
        ) as file:
            json.dump(
                fold_default_metrics,
                file,
                ensure_ascii=False,
                indent=2,
            )

        print("\nFold metrics at threshold 0.5:")
        print(json.dumps(
            {
                key: value
                for key, value in fold_default_metrics.items()
                if key not in {"classification_report"}
            },
            ensure_ascii=False,
            indent=2,
        ))

        del model, train_loader, valid_loader, adaptation_frame
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    # -------------------------------------------------------------------------
    # OOF threshold tuning.
    # -------------------------------------------------------------------------
    valid_oof_mask = ~np.isnan(oof_prob)

    if not valid_oof_mask.any():
        raise RuntimeError("No OOF predictions were generated.")

    official_null_labels = official_null["label"].to_numpy()
    oof_true = official_null_labels[valid_oof_mask]
    oof_probs = oof_prob[valid_oof_mask]

    default_metrics = evaluate_at_threshold(
        y_true=oof_true,
        prob_fair=oof_probs,
        threshold=0.5,
    )

    tuned_metrics = tune_threshold(
        y_true=oof_true,
        prob_fair=oof_probs,
        minimum=cfg.threshold_min,
        maximum=cfg.threshold_max,
        step=cfg.threshold_step,
    )

    print("\n" + "=" * 88)
    print("NULL-ROUTE OOF RESULTS")
    print("=" * 88)
    print("\nDefault threshold 0.5:")
    print(json.dumps(
        {
            key: value
            for key, value in default_metrics.items()
            if key != "classification_report"
        },
        ensure_ascii=False,
        indent=2,
    ))

    print("\nTuned threshold:")
    print(json.dumps(
        {
            key: value
            for key, value in tuned_metrics.items()
            if key != "classification_report"
        },
        ensure_ascii=False,
        indent=2,
    ))

    oof_output = official_null.copy()
    oof_output["prob_fair"] = oof_prob
    oof_output["pred_default"] = np.where(
        np.isnan(oof_prob),
        -1,
        (oof_prob >= 0.5).astype(int),
    )
    best_threshold = float(tuned_metrics["threshold"])
    oof_output["pred_tuned"] = np.where(
        np.isnan(oof_prob),
        -1,
        (oof_prob >= best_threshold).astype(int),
    )
    oof_output["oof_fold"] = oof_fold

    oof_output.to_csv(
        output_dir / "null_oof_predictions.csv",
        index=False,
    )

    # -------------------------------------------------------------------------
    # Ensemble null-route test probabilities.
    # -------------------------------------------------------------------------
    if not test_fold_probabilities:
        raise RuntimeError("No test probabilities were generated.")

    test_prob_matrix = np.vstack(test_fold_probabilities)
    test_prob_mean = test_prob_matrix.mean(axis=0)
    test_prob_std = test_prob_matrix.std(axis=0)

    test_null_output = test_null.copy().reset_index(drop=False).rename(
        columns={"index": "_test_row_index"}
    )
    test_null_output["prob_fair"] = test_prob_mean
    test_null_output["prob_std"] = test_prob_std
    test_null_output["pred_null_model"] = (
        test_prob_mean >= best_threshold
    ).astype(int)

    test_null_output.to_csv(
        output_dir / "null_test_predictions.csv",
        index=False,
    )

    np.save(
        output_dir / "null_test_fold_probabilities.npy",
        test_prob_matrix,
    )

    full_metrics = {
        "config": asdict(cfg),
        "fold_metrics": fold_metrics,
        "oof_default_threshold": default_metrics,
        "oof_tuned_threshold": tuned_metrics,
        "n_oof_rows": int(valid_oof_mask.sum()),
        "n_test_null_rows": int(len(test_null_output)),
    }

    with open(
        output_dir / "null_model_metrics.json",
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            full_metrics,
            file,
            ensure_ascii=False,
            indent=2,
        )

    print("\nSaved outputs:")
    for path in sorted(output_dir.iterdir()):
        print(" -", path)

    print("\nNext important files:")
    print("OOF:", output_dir / "null_oof_predictions.csv")
    print("Test:", output_dir / "null_test_predictions.csv")
    print("Metrics:", output_dir / "null_model_metrics.json")


if __name__ == "__main__":
    main()


### Step 3 — Null retrieval ensemble


In [ ]:

"""
STEP 3 — External dataset retrieval ensemble for NULL-CONTEXT rows.

Uses:
- External positive null-context QA rows as an offline knowledge base.
- Character n-gram TF-IDF for Bengali prompt retrieval.
- Character n-gram TF-IDF for response-vs-correct-answer similarity.
- Logistic-regression stacking over:
    BanglaBERT probability + retrieval features.
- 5-fold meta OOF evaluation on the 169 official null-context rows.
- Final prediction for the 1,155 null-context test rows.

Expected files:
    /kaggle/working/bh_preprocessed/final_train_max.parquet
    /kaggle/working/null_model_step2/null_oof_predictions.csv
    /kaggle/working/null_model_step2/null_test_predictions_macro.csv

Outputs:
    /kaggle/working/null_retrieval_step3/
"""

from __future__ import annotations

import json
import os
import re
import unicodedata
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.neighbors import NearestNeighbors


# =============================================================================
# CONFIG
# =============================================================================

PREPROCESSED_DIR = Path("/kaggle/working/bh_preprocessed")
NULL_MODEL_DIR = Path("/kaggle/working/null_model_step2")
OUTPUT_DIR = Path("/kaggle/working/null_retrieval_step3")

TRAIN_POOL_PATH = PREPROCESSED_DIR / "final_train_max.parquet"
OOF_PATH = NULL_MODEL_DIR / "null_oof_predictions.csv"
TEST_PATH = NULL_MODEL_DIR / "null_test_predictions.csv"

TOP_K = 5
RANDOM_STATE = 42

# Character n-grams are robust for Bengali spelling variants.
PROMPT_MAX_FEATURES = 250_000
ANSWER_MAX_FEATURES = 180_000


# =============================================================================
# TEXT HELPERS
# =============================================================================

NULL_STRINGS = {
    "", "nan", "none", "null", "nil", "n/a", "na", "[]", "{}",
    "[null]", '[""]', "()", "<na>",
}


def clean_text(value: object) -> str:
    if value is None:
        return ""

    try:
        if pd.isna(value):
            return ""
    except (TypeError, ValueError):
        pass

    text = unicodedata.normalize("NFC", str(value))
    text = text.replace("\u200b", " ").replace("\u200c", " ").replace("\u200d", " ")
    text = re.sub(r"\s+", " ", text).strip()

    if text.lower() in NULL_STRINGS:
        return ""

    return text


def normalize_series(series: pd.Series) -> pd.Series:
    return series.map(clean_text)


def is_official_source(value: object) -> bool:
    text = clean_text(value).lower()
    return any(token in text for token in ("official", "competition", "provided_sample"))


# =============================================================================
# DATA
# =============================================================================

def load_data() -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    for path in (TRAIN_POOL_PATH, OOF_PATH, TEST_PATH):
        if not path.exists():
            raise FileNotFoundError(f"Required file not found: {path}")

    pool = pd.read_parquet(TRAIN_POOL_PATH)
    oof = pd.read_csv(OOF_PATH, keep_default_na=False)
    test = pd.read_csv(TEST_PATH, keep_default_na=False)

    for frame in (pool, oof, test):
        for column in ("prompt_bn", "response_bn"):
            frame[column] = normalize_series(frame[column])

    if "route" in pool.columns:
        pool["route"] = (
            pool["route"]
            .fillna("null")
            .astype(str)
            .str.strip()
            .str.lower()
            .replace({"nan": "null", "none": "null", "": "null"})
        )

    # Retrieval KB = external, null-context, positive/correct answers only.
    kb = pool[
        pool["route"].eq("null")
        & pool["label"].eq(1)
    ].copy()

    if "source" in kb.columns:
        kb = kb[~kb["source"].map(is_official_source)].copy()

    kb = kb[
        kb["prompt_bn"].ne("")
        & kb["response_bn"].ne("")
    ].copy()

    # Keep distinct correct answers for each prompt.
    kb = kb.drop_duplicates(
        subset=["prompt_bn", "response_bn"]
    ).reset_index(drop=True)

    if "prob_fair" not in oof.columns:
        raise ValueError("OOF file does not contain prob_fair.")

    if "prob_fair" not in test.columns:
        raise ValueError("Test prediction file does not contain prob_fair.")

    if "label" not in oof.columns:
        raise ValueError("OOF file does not contain label.")

    print("Knowledge-base rows:", len(kb))
    print("Unique KB prompts:", kb["prompt_bn"].nunique())
    print("Official OOF rows:", len(oof))
    print("Null test rows:", len(test))

    return kb, oof, test


# =============================================================================
# RETRIEVAL
# =============================================================================

def build_retrieval_models(kb: pd.DataFrame):
    print("\nBuilding prompt TF-IDF...")

    prompt_vectorizer = TfidfVectorizer(
        analyzer="char_wb",
        ngram_range=(3, 5),
        min_df=2,
        max_features=PROMPT_MAX_FEATURES,
        sublinear_tf=True,
        norm="l2",
        dtype=np.float32,
    )

    kb_prompt_matrix = prompt_vectorizer.fit_transform(
        kb["prompt_bn"].tolist()
    )

    print("Prompt matrix:", kb_prompt_matrix.shape)

    prompt_nn = NearestNeighbors(
        n_neighbors=TOP_K,
        metric="cosine",
        algorithm="brute",
        n_jobs=-1,
    )
    prompt_nn.fit(kb_prompt_matrix)

    print("\nBuilding answer TF-IDF...")

    answer_vectorizer = TfidfVectorizer(
        analyzer="char_wb",
        ngram_range=(3, 5),
        min_df=2,
        max_features=ANSWER_MAX_FEATURES,
        sublinear_tf=True,
        norm="l2",
        dtype=np.float32,
    )

    kb_answer_matrix = answer_vectorizer.fit_transform(
        kb["response_bn"].tolist()
    )

    print("Answer matrix:", kb_answer_matrix.shape)

    return (
        prompt_vectorizer,
        prompt_nn,
        answer_vectorizer,
        kb_answer_matrix,
    )


def rowwise_sparse_cosine(
    query_matrix: sparse.csr_matrix,
    candidate_matrix: sparse.csr_matrix,
) -> np.ndarray:
    # Both matrices are L2-normalized by TF-IDF, so row-wise dot = cosine.
    return np.asarray(
        query_matrix.multiply(candidate_matrix).sum(axis=1)
    ).ravel().astype(np.float32)


def make_retrieval_features(
    frame: pd.DataFrame,
    kb: pd.DataFrame,
    prompt_vectorizer,
    prompt_nn,
    answer_vectorizer,
    kb_answer_matrix,
) -> pd.DataFrame:
    query_prompt_matrix = prompt_vectorizer.transform(
        frame["prompt_bn"].tolist()
    )

    distances, indices = prompt_nn.kneighbors(
        query_prompt_matrix,
        n_neighbors=TOP_K,
        return_distance=True,
    )

    prompt_similarities = 1.0 - distances
    query_answer_matrix = answer_vectorizer.transform(
        frame["response_bn"].tolist()
    )

    answer_similarity_columns: List[np.ndarray] = []

    for rank in range(TOP_K):
        candidate_indices = indices[:, rank]
        candidate_answer_matrix = kb_answer_matrix[candidate_indices]
        answer_sim = rowwise_sparse_cosine(
            query_answer_matrix,
            candidate_answer_matrix,
        )
        answer_similarity_columns.append(answer_sim)

    answer_similarities = np.column_stack(answer_similarity_columns)

    # Weight answer agreement by prompt relevance.
    weights = np.clip(prompt_similarities, 1e-6, None)
    weighted_answer_similarity = (
        (answer_similarities * weights).sum(axis=1)
        / weights.sum(axis=1)
    )

    best_joint = np.max(
        prompt_similarities * answer_similarities,
        axis=1,
    )

    best_rank = np.argmax(
        prompt_similarities * answer_similarities,
        axis=1,
    )

    best_kb_indices = indices[
        np.arange(len(frame)),
        best_rank,
    ]

    output = frame.copy()

    output["retr_prompt_top1"] = prompt_similarities[:, 0]
    output["retr_prompt_top3_mean"] = prompt_similarities[:, :3].mean(axis=1)
    output["retr_prompt_top5_mean"] = prompt_similarities.mean(axis=1)

    output["retr_answer_top1"] = answer_similarities[:, 0]
    output["retr_answer_max"] = answer_similarities.max(axis=1)
    output["retr_answer_weighted"] = weighted_answer_similarity
    output["retr_joint_max"] = best_joint

    output["retrieved_prompt"] = kb.iloc[
        best_kb_indices
    ]["prompt_bn"].to_numpy()

    output["retrieved_answer"] = kb.iloc[
        best_kb_indices
    ]["response_bn"].to_numpy()

    if "source" in kb.columns:
        output["retrieved_source"] = kb.iloc[
            best_kb_indices
        ]["source"].astype(str).to_numpy()
    else:
        output["retrieved_source"] = ""

    # Useful binary features.
    output["retr_high_prompt"] = (
        output["retr_prompt_top1"] >= 0.85
    ).astype(np.float32)

    output["retr_high_answer"] = (
        output["retr_answer_max"] >= 0.75
    ).astype(np.float32)

    output["response_len"] = (
        output["response_bn"].str.len().clip(upper=1000) / 1000.0
    ).astype(np.float32)

    output["prompt_len"] = (
        output["prompt_bn"].str.len().clip(upper=1000) / 1000.0
    ).astype(np.float32)

    return output


# =============================================================================
# META ENSEMBLE
# =============================================================================

FEATURE_COLUMNS = [
    "prob_fair",
    "retr_prompt_top1",
    "retr_prompt_top3_mean",
    "retr_prompt_top5_mean",
    "retr_answer_top1",
    "retr_answer_max",
    "retr_answer_weighted",
    "retr_joint_max",
    "retr_high_prompt",
    "retr_high_answer",
    "response_len",
    "prompt_len",
]


def metrics_at_threshold(
    y_true: np.ndarray,
    prob_fair: np.ndarray,
    threshold: float,
) -> Dict[str, object]:
    pred = (prob_fair >= threshold).astype(int)

    return {
        "threshold": float(threshold),
        "accuracy": float(accuracy_score(y_true, pred)),
        "f1_label0": float(
            f1_score(y_true, pred, pos_label=0, zero_division=0)
        ),
        "f1_label1": float(
            f1_score(y_true, pred, pos_label=1, zero_division=0)
        ),
        "macro_f1": float(
            f1_score(y_true, pred, average="macro", zero_division=0)
        ),
        "confusion_matrix": confusion_matrix(
            y_true,
            pred,
            labels=[0, 1],
        ).tolist(),
    }


def tune_macro_threshold(
    y_true: np.ndarray,
    probabilities: np.ndarray,
) -> Dict[str, object]:
    best = None

    for threshold in np.arange(0.05, 0.951, 0.005):
        result = metrics_at_threshold(
            y_true,
            probabilities,
            float(threshold),
        )

        key = (
            result["macro_f1"],
            result["accuracy"],
            result["f1_label0"],
            result["f1_label1"],
        )

        if best is None:
            best = result
            best_key = key
        elif key > best_key:
            best = result
            best_key = key

    return best


def run_meta_ensemble(
    oof_features: pd.DataFrame,
    test_features: pd.DataFrame,
):
    X = (
        oof_features[FEATURE_COLUMNS]
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0.0)
        .to_numpy(dtype=np.float32)
    )
    y = oof_features["label"].astype(int).to_numpy()

    X_test = (
        test_features[FEATURE_COLUMNS]
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0.0)
        .to_numpy(dtype=np.float32)
    )

    splitter = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=RANDOM_STATE,
    )

    meta_model = LogisticRegression(
        C=0.5,
        class_weight="balanced",
        max_iter=3000,
        solver="liblinear",
        random_state=RANDOM_STATE,
    )

    meta_oof_prob = cross_val_predict(
        meta_model,
        X,
        y,
        cv=splitter,
        method="predict_proba",
        n_jobs=-1,
    )[:, 1]

    baseline_metrics = tune_macro_threshold(
        y_true=y,
        probabilities=oof_features["prob_fair"].to_numpy(dtype=np.float32),
    )

    meta_metrics = tune_macro_threshold(
        y_true=y,
        probabilities=meta_oof_prob,
    )

    print("\nBaseline null-model metrics:")
    print(json.dumps(baseline_metrics, indent=2))

    print("\nRetrieval ensemble OOF metrics:")
    print(json.dumps(meta_metrics, indent=2))

    meta_model.fit(X, y)
    test_meta_prob = meta_model.predict_proba(X_test)[:, 1]

    return (
        meta_model,
        meta_oof_prob,
        test_meta_prob,
        baseline_metrics,
        meta_metrics,
    )


# =============================================================================
# MAIN
# =============================================================================

def main() -> None:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    print("=" * 88)
    print("STEP 3 — NULL-CONTEXT OFFLINE RETRIEVAL ENSEMBLE")
    print("=" * 88)

    kb, oof, test = load_data()

    (
        prompt_vectorizer,
        prompt_nn,
        answer_vectorizer,
        kb_answer_matrix,
    ) = build_retrieval_models(kb)

    print("\nGenerating official OOF retrieval features...")
    oof_features = make_retrieval_features(
        frame=oof,
        kb=kb,
        prompt_vectorizer=prompt_vectorizer,
        prompt_nn=prompt_nn,
        answer_vectorizer=answer_vectorizer,
        kb_answer_matrix=kb_answer_matrix,
    )

    print("Generating null-test retrieval features...")
    test_features = make_retrieval_features(
        frame=test,
        kb=kb,
        prompt_vectorizer=prompt_vectorizer,
        prompt_nn=prompt_nn,
        answer_vectorizer=answer_vectorizer,
        kb_answer_matrix=kb_answer_matrix,
    )

    (
        meta_model,
        meta_oof_prob,
        test_meta_prob,
        baseline_metrics,
        meta_metrics,
    ) = run_meta_ensemble(
        oof_features=oof_features,
        test_features=test_features,
    )

    best_threshold = float(meta_metrics["threshold"])

    oof_features["retrieval_ensemble_prob_fair"] = meta_oof_prob
    oof_features["retrieval_ensemble_pred"] = (
        meta_oof_prob >= best_threshold
    ).astype(int)

    test_features["retrieval_ensemble_prob_fair"] = test_meta_prob
    test_features["pred_retrieval_ensemble"] = (
        test_meta_prob >= best_threshold
    ).astype(int)

    oof_features.to_csv(
        OUTPUT_DIR / "null_retrieval_oof.csv",
        index=False,
    )

    test_features.to_csv(
        OUTPUT_DIR / "null_retrieval_test.csv",
        index=False,
    )

    metrics = {
        "knowledge_base_rows": int(len(kb)),
        "knowledge_base_unique_prompts": int(kb["prompt_bn"].nunique()),
        "top_k": int(TOP_K),
        "feature_columns": FEATURE_COLUMNS,
        "baseline_metrics": baseline_metrics,
        "retrieval_ensemble_metrics": meta_metrics,
        "meta_coefficients": {
            feature: float(coef)
            for feature, coef in zip(
                FEATURE_COLUMNS,
                meta_model.coef_[0],
            )
        },
        "meta_intercept": float(meta_model.intercept_[0]),
        "test_prediction_counts": {
            str(key): int(value)
            for key, value in test_features[
                "pred_retrieval_ensemble"
            ].value_counts().to_dict().items()
        },
    }

    with open(
        OUTPUT_DIR / "null_retrieval_metrics.json",
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            metrics,
            file,
            ensure_ascii=False,
            indent=2,
        )

    print("\n" + "=" * 88)
    print("STEP 3 COMPLETE")
    print("=" * 88)

    print(json.dumps(metrics, ensure_ascii=False, indent=2))

    print("\nSaved:")
    print(OUTPUT_DIR / "null_retrieval_oof.csv")
    print(OUTPUT_DIR / "null_retrieval_test.csv")
    print(OUTPUT_DIR / "null_retrieval_metrics.json")


if __name__ == "__main__":
    main()


### Step 4 — Internal context evidence verifier


In [ ]:

"""
STEP 4 — Context-present model training for Bengali Hallucination Competition.

Label:
    0 = hallucinated / unsupported
    1 = fair / supported

Uses:
    - External context-present data from final_train_max.parquet
    - Official context-present rows for 5-fold adaptation and OOF validation
    - Competition context-present test rows for inference

Expected files:
    /kaggle/working/bh_preprocessed/final_train_max.parquet
    /kaggle/working/bh_preprocessed/official_folds.csv
    /kaggle/working/bh_preprocessed/competition_test.csv

Outputs:
    /kaggle/working/context_model_step4/
"""

from __future__ import annotations

import gc
import json
import math
import os
import random
import re
import shutil
import unicodedata
from contextlib import nullcontext
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from torch.nn.utils import clip_grad_norm_
from torch.optim import AdamW
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    get_linear_schedule_with_warmup,
)


# =============================================================================
# CONFIG
# =============================================================================

@dataclass
class Config:
    input_dir: str = "/kaggle/working/bh_preprocessed"
    output_dir: str = "/kaggle/working/context_model_step4"

    model_name: str = "csebuetnlp/banglabert"
    seed: int = 42
    n_folds: int = 5

    # Context rows may be long.
    max_length: int = 384

    # Shared external context pretraining.
    external_epochs: int = 1
    external_batch_size: int = 8
    external_grad_accum: int = 4
    external_learning_rate: float = 2e-5
    external_weight_decay: float = 0.01
    external_warmup_ratio: float = 0.06

    # Fold-specific official adaptation.
    official_repeat: int = 35
    adaptation_external_size: int = 5000
    adaptation_epochs: int = 4
    adaptation_batch_size: int = 8
    adaptation_grad_accum: int = 2
    adaptation_learning_rate: float = 8e-6
    adaptation_weight_decay: float = 0.01
    adaptation_warmup_ratio: float = 0.05

    eval_batch_size: int = 32
    num_workers: int = 2
    max_grad_norm: float = 1.0

    threshold_min: float = 0.05
    threshold_max: float = 0.95
    threshold_step: float = 0.005

    folds_to_run: Tuple[int, ...] = (0, 1, 2, 3, 4)

    debug: bool = False
    debug_external_rows: int = 12000


CFG = Config()


# =============================================================================
# TEXT / SEED
# =============================================================================

NULL_STRINGS = {
    "", "nan", "none", "null", "nil", "n/a", "na", "[]", "{}",
    "[null]", '[""]', "()", "<na>",
}


def seed_everything(seed: int) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True


def clean_text(value: object) -> str:
    if value is None:
        return ""

    try:
        if pd.isna(value):
            return ""
    except (TypeError, ValueError):
        pass

    text = unicodedata.normalize("NFC", str(value))
    text = text.replace("\u200b", " ").replace("\u200c", " ").replace("\u200d", " ")
    text = re.sub(r"\s+", " ", text).strip()

    if text.lower() in NULL_STRINGS:
        return ""

    return text


def normalize_frame(
    df: pd.DataFrame,
    require_label: bool,
) -> pd.DataFrame:
    df = df.copy()

    required = ["context", "prompt_bn", "response_bn"]
    if require_label:
        required.append("label")

    missing = [column for column in required if column not in df.columns]
    if missing:
        raise ValueError(
            f"Missing columns: {missing}. Available: {df.columns.tolist()}"
        )

    for column in ["context", "prompt_bn", "response_bn"]:
        df[column] = df[column].map(clean_text)

    if "label" in df.columns:
        df["label"] = pd.to_numeric(
            df["label"],
            errors="raise",
        ).astype(int)

        if not set(df["label"].unique()).issubset({0, 1}):
            raise ValueError(
                f"Unexpected labels: {sorted(df['label'].unique())}"
            )

    if "route" not in df.columns:
        df["route"] = np.where(
            df["context"].eq(""),
            "null",
            "context",
        )
    else:
        df["route"] = (
            df["route"]
            .fillna("null")
            .astype(str)
            .str.strip()
            .str.lower()
            .replace({
                "nan": "null",
                "none": "null",
                "": "null",
            })
        )

    return df


def is_official_source(source: object) -> bool:
    text = clean_text(source).lower()
    return any(
        token in text
        for token in ["official", "competition", "provided_sample"]
    )


# =============================================================================
# DATASET
# =============================================================================

class ContextPairDataset(Dataset):
    def __init__(
        self,
        frame: pd.DataFrame,
        tokenizer,
        max_length: int,
        with_labels: bool,
    ) -> None:
        self.contexts = frame["context"].astype(str).tolist()
        self.prompts = frame["prompt_bn"].astype(str).tolist()
        self.responses = frame["response_bn"].astype(str).tolist()
        self.labels = (
            frame["label"].astype(int).tolist()
            if with_labels
            else None
        )
        self.with_labels = with_labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self) -> int:
        return len(self.contexts)

    def __getitem__(self, index: int) -> Dict[str, object]:
        # First sequence: evidence/context.
        # Second sequence: question + candidate response.
        second_sequence = (
            f"প্রশ্ন: {self.prompts[index]} "
            f"উত্তর: {self.responses[index]}"
        )

        encoded = self.tokenizer(
            self.contexts[index],
            second_sequence,
            truncation="longest_first",
            max_length=self.max_length,
            padding=False,
        )

        if self.with_labels:
            encoded["labels"] = self.labels[index]

        return encoded


def make_loader(
    frame: pd.DataFrame,
    tokenizer,
    max_length: int,
    batch_size: int,
    shuffle: bool,
    num_workers: int,
    with_labels: bool,
) -> DataLoader:
    dataset = ContextPairDataset(
        frame=frame,
        tokenizer=tokenizer,
        max_length=max_length,
        with_labels=with_labels,
    )

    collator = DataCollatorWithPadding(
        tokenizer=tokenizer,
        padding="longest",
        return_tensors="pt",
    )

    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=num_workers,
        pin_memory=torch.cuda.is_available(),
        persistent_workers=(num_workers > 0),
        collate_fn=collator,
    )


# =============================================================================
# TRAIN / PREDICT
# =============================================================================

def autocast_context(device: torch.device):
    if device.type == "cuda":
        return torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
        )
    return nullcontext()


@torch.no_grad()
def predict_probabilities(
    model,
    loader: DataLoader,
    device: torch.device,
    return_labels: bool,
) -> Tuple[np.ndarray, Optional[np.ndarray]]:
    model.eval()
    model.to(device)

    probabilities: List[np.ndarray] = []
    labels_list: List[np.ndarray] = []

    for batch in tqdm(loader, desc="Predicting", leave=False):
        labels = batch.pop("labels", None)

        batch = {
            key: value.to(device, non_blocking=True)
            for key, value in batch.items()
        }

        with autocast_context(device):
            logits = model(**batch).logits

        prob_fair = torch.softmax(
            logits.float(),
            dim=-1,
        )[:, 1]

        probabilities.append(prob_fair.cpu().numpy())

        if labels is not None:
            labels_list.append(labels.numpy())

    probs = np.concatenate(probabilities)

    if return_labels:
        return probs, np.concatenate(labels_list)

    return probs, None


def train_model(
    model,
    train_loader: DataLoader,
    device: torch.device,
    epochs: int,
    learning_rate: float,
    weight_decay: float,
    warmup_ratio: float,
    grad_accum_steps: int,
    max_grad_norm: float,
    stage_name: str,
    validation_loader: Optional[DataLoader],
):
    optimizer = AdamW(
        model.parameters(),
        lr=learning_rate,
        weight_decay=weight_decay,
    )

    updates_per_epoch = math.ceil(
        len(train_loader) / grad_accum_steps
    )
    total_updates = max(
        1,
        updates_per_epoch * epochs,
    )
    warmup_steps = int(
        total_updates * warmup_ratio
    )

    scheduler = get_linear_schedule_with_warmup(
        optimizer=optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_updates,
    )

    scaler = torch.amp.GradScaler(
        "cuda",
        enabled=(device.type == "cuda"),
    )

    best_macro_f1 = -1.0
    best_state = None
    best_metrics: Dict[str, float] = {}

    model.to(device)

    for epoch in range(1, epochs + 1):
        model.train()
        optimizer.zero_grad(set_to_none=True)

        running_loss = 0.0
        batches_seen = 0

        progress = tqdm(
            train_loader,
            desc=f"{stage_name} | epoch {epoch}/{epochs}",
        )

        for step, batch in enumerate(progress, start=1):
            batch = {
                key: value.to(device, non_blocking=True)
                for key, value in batch.items()
            }

            with autocast_context(device):
                outputs = model(**batch)
                loss = outputs.loss / grad_accum_steps

            scaler.scale(loss).backward()

            should_update = (
                step % grad_accum_steps == 0
                or step == len(train_loader)
            )

            if should_update:
                scaler.unscale_(optimizer)
                clip_grad_norm_(
                    model.parameters(),
                    max_grad_norm,
                )
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)

            running_loss += (
                loss.item() * grad_accum_steps
            )
            batches_seen += 1

            progress.set_postfix(
                loss=f"{running_loss / batches_seen:.4f}",
                lr=f"{scheduler.get_last_lr()[0]:.2e}",
            )

        epoch_metrics = {
            "epoch": float(epoch),
            "train_loss": (
                running_loss / max(1, batches_seen)
            ),
        }

        if validation_loader is not None:
            val_prob, val_labels = predict_probabilities(
                model=model,
                loader=validation_loader,
                device=device,
                return_labels=True,
            )

            val_pred = (val_prob >= 0.5).astype(int)

            epoch_metrics.update({
                "val_accuracy": float(
                    accuracy_score(
                        val_labels,
                        val_pred,
                    )
                ),
                "val_f1_label0": float(
                    f1_score(
                        val_labels,
                        val_pred,
                        pos_label=0,
                        zero_division=0,
                    )
                ),
                "val_f1_label1": float(
                    f1_score(
                        val_labels,
                        val_pred,
                        pos_label=1,
                        zero_division=0,
                    )
                ),
                "val_macro_f1": float(
                    f1_score(
                        val_labels,
                        val_pred,
                        average="macro",
                        zero_division=0,
                    )
                ),
            })

            print(json.dumps(
                epoch_metrics,
                ensure_ascii=False,
                indent=2,
            ))

            if (
                epoch_metrics["val_macro_f1"]
                > best_macro_f1
            ):
                best_macro_f1 = (
                    epoch_metrics["val_macro_f1"]
                )
                best_metrics = epoch_metrics.copy()
                best_state = {
                    key: value.detach().cpu().clone()
                    for key, value
                    in model.state_dict().items()
                }

        else:
            print(json.dumps(
                epoch_metrics,
                ensure_ascii=False,
                indent=2,
            ))

    if best_state is not None:
        model.load_state_dict(best_state)

    return model, best_metrics


# =============================================================================
# METRICS
# =============================================================================

def evaluate_at_threshold(
    y_true: np.ndarray,
    prob_fair: np.ndarray,
    threshold: float,
) -> Dict[str, object]:
    prediction = (
        prob_fair >= threshold
    ).astype(int)

    return {
        "threshold": float(threshold),
        "accuracy": float(
            accuracy_score(
                y_true,
                prediction,
            )
        ),
        "f1_label0": float(
            f1_score(
                y_true,
                prediction,
                pos_label=0,
                zero_division=0,
            )
        ),
        "f1_label1": float(
            f1_score(
                y_true,
                prediction,
                pos_label=1,
                zero_division=0,
            )
        ),
        "macro_f1": float(
            f1_score(
                y_true,
                prediction,
                average="macro",
                zero_division=0,
            )
        ),
        "confusion_matrix": confusion_matrix(
            y_true,
            prediction,
            labels=[0, 1],
        ).tolist(),
        "classification_report": classification_report(
            y_true,
            prediction,
            labels=[0, 1],
            target_names=[
                "hallucinated_0",
                "fair_1",
            ],
            zero_division=0,
            output_dict=True,
        ),
    }


def tune_threshold(
    y_true: np.ndarray,
    prob_fair: np.ndarray,
    minimum: float,
    maximum: float,
    step: float,
) -> Dict[str, object]:
    best = None
    best_key = None

    for threshold in np.arange(
        minimum,
        maximum + 1e-9,
        step,
    ):
        metrics = evaluate_at_threshold(
            y_true=y_true,
            prob_fair=prob_fair,
            threshold=float(threshold),
        )

        current_key = (
            metrics["macro_f1"],
            metrics["accuracy"],
            metrics["f1_label0"],
            metrics["f1_label1"],
        )

        if (
            best is None
            or current_key > best_key
        ):
            best = metrics
            best_key = current_key

    return best


# =============================================================================
# DATA PREPARATION
# =============================================================================

def load_data(cfg: Config):
    input_dir = Path(cfg.input_dir)

    train_path = (
        input_dir / "final_train_max.parquet"
    )
    official_path = (
        input_dir / "official_folds.csv"
    )
    test_path = (
        input_dir / "competition_test.csv"
    )

    for path in [
        train_path,
        official_path,
        test_path,
    ]:
        if not path.exists():
            raise FileNotFoundError(
                f"Required file not found: {path}"
            )

    final_train = normalize_frame(
        pd.read_parquet(train_path),
        require_label=True,
    )

    official = normalize_frame(
        pd.read_csv(
            official_path,
            keep_default_na=False,
        ),
        require_label=True,
    )

    test = normalize_frame(
        pd.read_csv(
            test_path,
            keep_default_na=False,
        ),
        require_label=False,
    )

    official["fold"] = pd.to_numeric(
        official["fold"],
        errors="raise",
    ).astype(int)

    context_pool = final_train[
        final_train["route"].eq("context")
    ].copy()

    if "source" in context_pool.columns:
        external_context = context_pool[
            ~context_pool["source"].map(
                is_official_source
            )
        ].copy()
    else:
        official_keys = set(zip(
            official["context"],
            official["prompt_bn"],
            official["response_bn"],
        ))

        mask = [
            (
                context,
                prompt,
                response,
            ) not in official_keys
            for context, prompt, response
            in zip(
                context_pool["context"],
                context_pool["prompt_bn"],
                context_pool["response_bn"],
            )
        ]

        external_context = context_pool[
            np.asarray(mask)
        ].copy()

    official_context = official[
        official["route"].eq("context")
    ].copy()

    test_context = test[
        test["route"].eq("context")
    ].copy()

    return (
        external_context,
        official_context,
        test,
        test_context,
    )


def balanced_sample(
    frame: pd.DataFrame,
    total_size: int,
    seed: int,
) -> pd.DataFrame:
    if len(frame) <= total_size:
        return frame.sample(
            frac=1,
            random_state=seed,
        ).reset_index(drop=True)

    per_label = total_size // 2
    parts = []

    for label in [0, 1]:
        group = frame[
            frame["label"].eq(label)
        ]

        n = min(
            per_label,
            len(group),
        )

        parts.append(
            group.sample(
                n=n,
                random_state=seed + label,
            )
        )

    result = pd.concat(
        parts,
        ignore_index=True,
    )

    return result.sample(
        frac=1,
        random_state=seed,
    ).reset_index(drop=True)


def make_adaptation_frame(
    official_train: pd.DataFrame,
    external_context: pd.DataFrame,
    cfg: Config,
    fold: int,
) -> pd.DataFrame:
    official_repeated = pd.concat(
        [official_train] * cfg.official_repeat,
        ignore_index=True,
    )

    official_repeated[
        "adaptation_origin"
    ] = "official"

    external_subset = balanced_sample(
        frame=external_context,
        total_size=cfg.adaptation_external_size,
        seed=cfg.seed + fold * 101,
    )

    external_subset[
        "adaptation_origin"
    ] = "external"

    combined = pd.concat(
        [
            official_repeated,
            external_subset,
        ],
        ignore_index=True,
    )

    return combined.sample(
        frac=1,
        random_state=cfg.seed + fold,
    ).reset_index(drop=True)


# =============================================================================
# MAIN
# =============================================================================

def main() -> None:
    cfg = CFG
    seed_everything(cfg.seed)

    output_dir = Path(cfg.output_dir)
    output_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    with open(
        output_dir / "config.json",
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            asdict(cfg),
            file,
            ensure_ascii=False,
            indent=2,
        )

    device = torch.device(
        "cuda"
        if torch.cuda.is_available()
        else "cpu"
    )

    print("=" * 88)
    print("STEP 4 — CONTEXT-PRESENT MODEL")
    print("=" * 88)
    print("Device:", device)

    if device.type == "cuda":
        print(
            "GPU:",
            torch.cuda.get_device_name(0),
        )

    (
        external_context,
        official_context,
        full_test,
        test_context,
    ) = load_data(cfg)

    if cfg.debug:
        external_context = balanced_sample(
            external_context,
            total_size=min(
                cfg.debug_external_rows,
                len(external_context),
            ),
            seed=cfg.seed,
        )

    print("\n[DATA]")
    print(
        "External context rows:",
        len(external_context),
    )
    print(
        "Official context rows:",
        len(official_context),
    )
    print(
        "Test context rows:",
        len(test_context),
    )

    print("\nExternal labels:")
    print(
        external_context[
            "label"
        ].value_counts()
    )

    print("\nOfficial context fold/label:")
    print(pd.crosstab(
        official_context["fold"],
        official_context["label"],
        margins=True,
    ))

    tokenizer = AutoTokenizer.from_pretrained(
        cfg.model_name,
        use_fast=True,
    )

    # -------------------------------------------------------------------------
    # Shared external context base.
    # -------------------------------------------------------------------------
    external_model_dir = (
        output_dir / "external_context_base"
    )

    if (
        external_model_dir / "config.json"
    ).exists():
        print(
            "\n[STAGE A] Existing external "
            "context base found."
        )
    else:
        print(
            "\n[STAGE A] Training shared "
            "external context base."
        )

        model = (
            AutoModelForSequenceClassification
            .from_pretrained(
                cfg.model_name,
                num_labels=2,
                id2label={
                    0: "HALLUCINATED",
                    1: "FAIR",
                },
                label2id={
                    "HALLUCINATED": 0,
                    "FAIR": 1,
                },
            )
        )

        external_loader = make_loader(
            frame=external_context,
            tokenizer=tokenizer,
            max_length=cfg.max_length,
            batch_size=cfg.external_batch_size,
            shuffle=True,
            num_workers=cfg.num_workers,
            with_labels=True,
        )

        model, _ = train_model(
            model=model,
            train_loader=external_loader,
            device=device,
            epochs=cfg.external_epochs,
            learning_rate=(
                cfg.external_learning_rate
            ),
            weight_decay=(
                cfg.external_weight_decay
            ),
            warmup_ratio=(
                cfg.external_warmup_ratio
            ),
            grad_accum_steps=(
                cfg.external_grad_accum
            ),
            max_grad_norm=(
                cfg.max_grad_norm
            ),
            stage_name=(
                "External context pretraining"
            ),
            validation_loader=None,
        )

        external_model_dir.mkdir(
            parents=True,
            exist_ok=True,
        )

        model.save_pretrained(
            external_model_dir
        )
        tokenizer.save_pretrained(
            external_model_dir
        )

        del model, external_loader
        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    # -------------------------------------------------------------------------
    # Fold adaptation.
    # -------------------------------------------------------------------------
    official_context = (
        official_context
        .reset_index(drop=True)
    )

    official_context[
        "_official_context_index"
    ] = np.arange(
        len(official_context),
        dtype=np.int64,
    )

    oof_prob = np.full(
        len(official_context),
        np.nan,
        dtype=np.float32,
    )

    oof_fold = np.full(
        len(official_context),
        -1,
        dtype=np.int32,
    )

    test_fold_probabilities = []
    fold_metrics = []

    test_context_loader = make_loader(
        frame=test_context,
        tokenizer=tokenizer,
        max_length=cfg.max_length,
        batch_size=cfg.eval_batch_size,
        shuffle=False,
        num_workers=cfg.num_workers,
        with_labels=False,
    )

    for fold in cfg.folds_to_run:
        print("\n" + "=" * 88)
        print(f"FOLD {fold}")
        print("=" * 88)

        fold_train = official_context[
            official_context["fold"].ne(fold)
        ].copy()

        fold_valid = official_context[
            official_context["fold"].eq(fold)
        ].copy()

        if fold_valid.empty:
            print(
                f"[WARN] Fold {fold} has no "
                "context validation rows."
            )
            continue

        adaptation_frame = make_adaptation_frame(
            official_train=fold_train,
            external_context=external_context,
            cfg=cfg,
            fold=fold,
        )

        print(
            "Official train rows:",
            len(fold_train),
        )
        print(
            "Official validation rows:",
            len(fold_valid),
        )
        print(
            "Adaptation rows:",
            len(adaptation_frame),
        )

        print(pd.crosstab(
            adaptation_frame[
                "adaptation_origin"
            ],
            adaptation_frame["label"],
            margins=True,
        ))

        model = (
            AutoModelForSequenceClassification
            .from_pretrained(
                external_model_dir,
                num_labels=2,
            )
        )

        train_loader = make_loader(
            frame=adaptation_frame,
            tokenizer=tokenizer,
            max_length=cfg.max_length,
            batch_size=(
                cfg.adaptation_batch_size
            ),
            shuffle=True,
            num_workers=cfg.num_workers,
            with_labels=True,
        )

        valid_loader = make_loader(
            frame=fold_valid,
            tokenizer=tokenizer,
            max_length=cfg.max_length,
            batch_size=cfg.eval_batch_size,
            shuffle=False,
            num_workers=cfg.num_workers,
            with_labels=True,
        )

        model, best_epoch_metrics = train_model(
            model=model,
            train_loader=train_loader,
            device=device,
            epochs=cfg.adaptation_epochs,
            learning_rate=(
                cfg.adaptation_learning_rate
            ),
            weight_decay=(
                cfg.adaptation_weight_decay
            ),
            warmup_ratio=(
                cfg.adaptation_warmup_ratio
            ),
            grad_accum_steps=(
                cfg.adaptation_grad_accum
            ),
            max_grad_norm=(
                cfg.max_grad_norm
            ),
            stage_name=(
                f"Official context adaptation "
                f"fold {fold}"
            ),
            validation_loader=valid_loader,
        )

        fold_valid_prob, fold_valid_labels = (
            predict_probabilities(
                model=model,
                loader=valid_loader,
                device=device,
                return_labels=True,
            )
        )

        positions = fold_valid[
            "_official_context_index"
        ].to_numpy(dtype=np.int64)

        oof_prob[positions] = (
            fold_valid_prob
        )
        oof_fold[positions] = fold

        fold_test_prob, _ = (
            predict_probabilities(
                model=model,
                loader=test_context_loader,
                device=device,
                return_labels=False,
            )
        )

        test_fold_probabilities.append(
            fold_test_prob
        )

        fold_default_metrics = (
            evaluate_at_threshold(
                y_true=fold_valid_labels,
                prob_fair=fold_valid_prob,
                threshold=0.5,
            )
        )

        fold_default_metrics[
            "fold"
        ] = int(fold)

        fold_default_metrics[
            "best_training_epoch"
        ] = best_epoch_metrics

        fold_metrics.append(
            fold_default_metrics
        )

        fold_model_dir = (
            output_dir / f"fold_{fold}"
        )

        fold_model_dir.mkdir(
            parents=True,
            exist_ok=True,
        )

        model.save_pretrained(
            fold_model_dir
        )
        tokenizer.save_pretrained(
            fold_model_dir
        )

        with open(
            fold_model_dir / "fold_metrics.json",
            "w",
            encoding="utf-8",
        ) as file:
            json.dump(
                fold_default_metrics,
                file,
                ensure_ascii=False,
                indent=2,
            )

        print(
            "\nFold metrics at threshold 0.5:"
        )
        print(json.dumps(
            {
                key: value
                for key, value
                in fold_default_metrics.items()
                if key != "classification_report"
            },
            ensure_ascii=False,
            indent=2,
        ))

        del (
            model,
            train_loader,
            valid_loader,
            adaptation_frame,
        )

        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    # -------------------------------------------------------------------------
    # OOF threshold.
    # -------------------------------------------------------------------------
    valid_mask = ~np.isnan(oof_prob)

    if not valid_mask.any():
        raise RuntimeError(
            "No context OOF predictions generated."
        )

    labels = official_context[
        "label"
    ].to_numpy()

    y_true = labels[valid_mask]
    probs = oof_prob[valid_mask]

    default_metrics = evaluate_at_threshold(
        y_true=y_true,
        prob_fair=probs,
        threshold=0.5,
    )

    tuned_metrics = tune_threshold(
        y_true=y_true,
        prob_fair=probs,
        minimum=cfg.threshold_min,
        maximum=cfg.threshold_max,
        step=cfg.threshold_step,
    )

    print("\n" + "=" * 88)
    print("CONTEXT-ROUTE OOF RESULTS")
    print("=" * 88)

    print("\nDefault threshold:")
    print(json.dumps(
        {
            key: value
            for key, value
            in default_metrics.items()
            if key != "classification_report"
        },
        ensure_ascii=False,
        indent=2,
    ))

    print("\nTuned threshold:")
    print(json.dumps(
        {
            key: value
            for key, value
            in tuned_metrics.items()
            if key != "classification_report"
        },
        ensure_ascii=False,
        indent=2,
    ))

    best_threshold = float(
        tuned_metrics["threshold"]
    )

    oof_output = official_context.copy()
    oof_output["prob_fair"] = oof_prob
    oof_output["pred_default"] = np.where(
        np.isnan(oof_prob),
        -1,
        (oof_prob >= 0.5).astype(int),
    )
    oof_output["pred_tuned"] = np.where(
        np.isnan(oof_prob),
        -1,
        (
            oof_prob >= best_threshold
        ).astype(int),
    )
    oof_output["oof_fold"] = oof_fold

    oof_output.to_csv(
        output_dir
        / "context_oof_predictions.csv",
        index=False,
    )

    # -------------------------------------------------------------------------
    # Test ensemble.
    # -------------------------------------------------------------------------
    if not test_fold_probabilities:
        raise RuntimeError(
            "No context test probabilities."
        )

    test_probability_matrix = np.vstack(
        test_fold_probabilities
    )

    test_prob_mean = (
        test_probability_matrix.mean(axis=0)
    )
    test_prob_std = (
        test_probability_matrix.std(axis=0)
    )

    test_context_output = (
        test_context
        .copy()
        .reset_index(drop=False)
        .rename(
            columns={
                "index": "_test_row_index"
            }
        )
    )

    test_context_output[
        "prob_fair"
    ] = test_prob_mean

    test_context_output[
        "prob_std"
    ] = test_prob_std

    test_context_output[
        "pred_context_model"
    ] = (
        test_prob_mean >= best_threshold
    ).astype(int)

    test_context_output.to_csv(
        output_dir
        / "context_test_predictions.csv",
        index=False,
    )

    np.save(
        output_dir
        / "context_test_fold_probabilities.npy",
        test_probability_matrix,
    )

    full_metrics = {
        "config": asdict(cfg),
        "fold_metrics": fold_metrics,
        "oof_default_threshold": (
            default_metrics
        ),
        "oof_tuned_threshold": (
            tuned_metrics
        ),
        "n_oof_rows": int(
            valid_mask.sum()
        ),
        "n_test_context_rows": int(
            len(test_context_output)
        ),
    }

    with open(
        output_dir
        / "context_model_metrics.json",
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            full_metrics,
            file,
            ensure_ascii=False,
            indent=2,
        )

    print("\nSaved outputs:")
    for path in sorted(
        output_dir.iterdir()
    ):
        print(" -", path)


if __name__ == "__main__":
    main()


### Step 6 — Hybrid null verifier


In [ ]:

""" 
STEP 6 — HYBRID RETRIEVAL + TOP-5 EVIDENCE VERIFIER FOR NULL-CONTEXT ROWS

Goal
----
Improve the current null-context system by:

1. Building a larger offline knowledge base from:
   - positive/correct null-context external rows
   - positive/supported context-present external rows

2. Retrieving evidence with two lexical channels:
   - character n-gram TF-IDF
   - word n-gram TF-IDF

3. Fusing both rankings with Reciprocal Rank Fusion (RRF).

4. Sending the top-5 retrieved evidence passages through the already-trained
   context specialist models:
       /kaggle/working/context_model_step4/fold_0 ... fold_4

5. Combining:
   - current null retrieval probability
   - original null neural probability
   - top-5 verifier probabilities
   - retrieval confidence features

   using an OOF-safe Logistic Regression meta-model.

Inputs
------
/kaggle/working/bh_preprocessed/final_train_max.parquet
/kaggle/working/null_retrieval_step3/null_retrieval_oof.csv
/kaggle/working/null_retrieval_step3/null_retrieval_test.csv
/kaggle/working/context_model_step4/fold_0 ... fold_4

Outputs
-------
/kaggle/working/null_step6_hybrid_verifier/
    step6_null_oof.csv
    step6_null_test.csv
    step6_metrics.json
    step6_retrieved_pairs_oof.csv
    step6_retrieved_pairs_test.csv

Important
---------
The official null OOF rows are NOT inserted into the knowledge base.
Only external positive rows are used as evidence.
"""

from __future__ import annotations

import gc
import json
import os
import random
import re
import unicodedata
from contextlib import nullcontext
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
import torch
from scipy import sparse
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.neighbors import NearestNeighbors
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
)


# =============================================================================
# CONFIG
# =============================================================================

SEED = 42

PREPROCESSED_DIR = Path("/kaggle/working/bh_preprocessed")
NULL_RETRIEVAL_DIR = Path("/kaggle/working/null_retrieval_step3")
CONTEXT_MODEL_DIR = Path("/kaggle/working/context_model_step4")
OUTPUT_DIR = Path("/kaggle/working/null_step6_hybrid_verifier")

TRAIN_POOL_PATH = PREPROCESSED_DIR / "final_train_max.parquet"
NULL_OOF_PATH = NULL_RETRIEVAL_DIR / "null_retrieval_oof.csv"
NULL_TEST_PATH = NULL_RETRIEVAL_DIR / "null_retrieval_test.csv"

TOP_CANDIDATES_PER_CHANNEL = 20
TOP_EVIDENCE = 5
RRF_K = 60.0

CHAR_MAX_FEATURES = 230_000
WORD_MAX_FEATURES = 170_000

MAX_CONTEXT_CHARS = 2800
MAX_LENGTH = 384
VERIFY_BATCH_SIZE = 32
NUM_WORKERS = 2

META_C_VALUES = [0.05, 0.10, 0.25, 0.50, 1.00, 2.00]
BLEND_WEIGHTS = np.arange(0.0, 1.001, 0.05)
THRESHOLDS = np.arange(0.05, 0.951, 0.005)

FOLD_MODEL_DIRS = [
    CONTEXT_MODEL_DIR / f"fold_{fold}"
    for fold in range(5)
]


# =============================================================================
# UTILITIES
# =============================================================================

NULL_STRINGS = {
    "", "nan", "none", "null", "nil", "n/a", "na", "[]", "{}",
    "[null]", '[""]', "()", "<na>",
}


def seed_everything(seed: int) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True


def clean_text(value: object) -> str:
    if value is None:
        return ""

    try:
        if pd.isna(value):
            return ""
    except (TypeError, ValueError):
        pass

    text = unicodedata.normalize("NFC", str(value))
    text = (
        text.replace("\u200b", " ")
        .replace("\u200c", " ")
        .replace("\u200d", " ")
    )
    text = re.sub(r"\s+", " ", text).strip()

    if text.lower() in NULL_STRINGS:
        return ""

    return text


def normalize_route(value: object, context: object) -> str:
    route = clean_text(value).lower()

    if route in {"context", "context_present", "with_context"}:
        return "context"

    if route in {"null", "no_context", "without_context"}:
        return "null"

    return "null" if clean_text(context) == "" else "context"


def is_official_source(value: object) -> bool:
    source = clean_text(value).lower()

    return any(
        token in source
        for token in (
            "official",
            "competition",
            "provided_sample",
        )
    )


def metrics_at_threshold(
    y_true: np.ndarray,
    probability: np.ndarray,
    threshold: float,
) -> Dict[str, object]:
    prediction = (probability >= threshold).astype(int)

    return {
        "threshold": float(threshold),
        "accuracy": float(accuracy_score(y_true, prediction)),
        "f1_label0": float(
            f1_score(
                y_true,
                prediction,
                pos_label=0,
                zero_division=0,
            )
        ),
        "f1_label1": float(
            f1_score(
                y_true,
                prediction,
                pos_label=1,
                zero_division=0,
            )
        ),
        "macro_f1": float(
            f1_score(
                y_true,
                prediction,
                average="macro",
                zero_division=0,
            )
        ),
        "confusion_matrix": confusion_matrix(
            y_true,
            prediction,
            labels=[0, 1],
        ).tolist(),
    }


def tune_threshold(
    y_true: np.ndarray,
    probability: np.ndarray,
) -> Dict[str, object]:
    best: Optional[Dict[str, object]] = None
    best_key: Optional[Tuple[float, ...]] = None

    for threshold in THRESHOLDS:
        result = metrics_at_threshold(
            y_true=y_true,
            probability=probability,
            threshold=float(threshold),
        )

        key = (
            float(result["macro_f1"]),
            float(result["accuracy"]),
            float(result["f1_label0"]),
            float(result["f1_label1"]),
        )

        if best is None or key > best_key:
            best = result
            best_key = key

    assert best is not None
    return best


# =============================================================================
# DATA
# =============================================================================

def load_inputs() -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    required_paths = [
        TRAIN_POOL_PATH,
        NULL_OOF_PATH,
        NULL_TEST_PATH,
        *FOLD_MODEL_DIRS,
    ]

    for path in required_paths:
        if not path.exists():
            raise FileNotFoundError(f"Required input not found: {path}")

    pool = pd.read_parquet(TRAIN_POOL_PATH)
    oof = pd.read_csv(NULL_OOF_PATH, keep_default_na=False)
    test = pd.read_csv(NULL_TEST_PATH, keep_default_na=False)

    for frame in (pool, oof, test):
        for column in ("context", "prompt_bn", "response_bn"):
            if column not in frame.columns:
                frame[column] = ""
            frame[column] = frame[column].map(clean_text)

    if "route" not in pool.columns:
        pool["route"] = [
            normalize_route("", context)
            for context in pool["context"]
        ]
    else:
        pool["route"] = [
            normalize_route(route, context)
            for route, context in zip(
                pool["route"],
                pool["context"],
            )
        ]

    pool["label"] = pd.to_numeric(
        pool["label"],
        errors="raise",
    ).astype(int)

    if "label" not in oof.columns:
        raise ValueError("null_retrieval_oof.csv has no label column.")

    if "retrieval_ensemble_prob_fair" not in oof.columns:
        raise ValueError(
            "OOF file has no retrieval_ensemble_prob_fair column."
        )

    if "retrieval_ensemble_prob_fair" not in test.columns:
        raise ValueError(
            "Test file has no retrieval_ensemble_prob_fair column."
        )

    return pool, oof, test


def build_knowledge_base(pool: pd.DataFrame) -> pd.DataFrame:
    external = pool.copy()

    if "source" in external.columns:
        external = external[
            ~external["source"].map(is_official_source)
        ].copy()
    else:
        external["source"] = "external"

    positive = external[
        external["label"].eq(1)
        & external["prompt_bn"].ne("")
        & external["response_bn"].ne("")
    ].copy()

    null_rows = positive[
        positive["route"].eq("null")
    ].copy()

    context_rows = positive[
        positive["route"].eq("context")
        & positive["context"].ne("")
    ].copy()

    # For a null-QA row, synthesize a compact factual evidence sentence.
    null_kb = pd.DataFrame({
        "kb_prompt": null_rows["prompt_bn"],
        "kb_answer": null_rows["response_bn"],
        "kb_evidence": (
            "প্রশ্ন: "
            + null_rows["prompt_bn"]
            + " সঠিক উত্তর: "
            + null_rows["response_bn"]
        ),
        "kb_route": "null",
        "kb_source": null_rows["source"].astype(str),
    })

    # For a context row, preserve the original evidence and append the known
    # supported answer. This helps the verifier when the context is indirect.
    context_evidence = (
        context_rows["context"].str.slice(0, MAX_CONTEXT_CHARS)
        + " প্রাসঙ্গিক সঠিক উত্তর: "
        + context_rows["response_bn"]
    )

    context_kb = pd.DataFrame({
        "kb_prompt": context_rows["prompt_bn"],
        "kb_answer": context_rows["response_bn"],
        "kb_evidence": context_evidence,
        "kb_route": "context",
        "kb_source": context_rows["source"].astype(str),
    })

    kb = pd.concat(
        [null_kb, context_kb],
        ignore_index=True,
    )

    for column in (
        "kb_prompt",
        "kb_answer",
        "kb_evidence",
        "kb_source",
    ):
        kb[column] = kb[column].map(clean_text)

    kb = kb[
        kb["kb_prompt"].ne("")
        & kb["kb_evidence"].ne("")
    ].copy()

    kb = kb.drop_duplicates(
        subset=[
            "kb_prompt",
            "kb_answer",
            "kb_evidence",
        ]
    ).reset_index(drop=True)

    kb["kb_id"] = np.arange(
        len(kb),
        dtype=np.int64,
    )

    print("\n[KNOWLEDGE BASE]")
    print("Total evidence rows:", len(kb))
    print("Unique prompts:", kb["kb_prompt"].nunique())
    print("Route counts:")
    print(kb["kb_route"].value_counts())

    return kb


# =============================================================================
# HYBRID RETRIEVAL
# =============================================================================

def build_retrievers(kb: pd.DataFrame):
    print("\nBuilding character TF-IDF retriever...")

    char_vectorizer = TfidfVectorizer(
        analyzer="char_wb",
        ngram_range=(3, 5),
        min_df=2,
        max_features=CHAR_MAX_FEATURES,
        sublinear_tf=True,
        norm="l2",
        dtype=np.float32,
    )

    char_matrix = char_vectorizer.fit_transform(
        kb["kb_prompt"].tolist()
    )

    char_nn = NearestNeighbors(
        n_neighbors=TOP_CANDIDATES_PER_CHANNEL,
        metric="cosine",
        algorithm="brute",
        n_jobs=-1,
    )
    char_nn.fit(char_matrix)

    print("Character matrix:", char_matrix.shape)

    print("\nBuilding word TF-IDF retriever...")

    word_vectorizer = TfidfVectorizer(
        analyzer="word",
        token_pattern=r"(?u)[\u0980-\u09FFa-zA-Z0-9]+",
        ngram_range=(1, 2),
        min_df=2,
        max_features=WORD_MAX_FEATURES,
        sublinear_tf=True,
        norm="l2",
        dtype=np.float32,
    )

    word_matrix = word_vectorizer.fit_transform(
        kb["kb_prompt"].tolist()
    )

    word_nn = NearestNeighbors(
        n_neighbors=TOP_CANDIDATES_PER_CHANNEL,
        metric="cosine",
        algorithm="brute",
        n_jobs=-1,
    )
    word_nn.fit(word_matrix)

    print("Word matrix:", word_matrix.shape)

    return (
        char_vectorizer,
        char_nn,
        word_vectorizer,
        word_nn,
    )


def retrieve_hybrid(
    frame: pd.DataFrame,
    kb: pd.DataFrame,
    char_vectorizer,
    char_nn,
    word_vectorizer,
    word_nn,
    split_name: str,
) -> pd.DataFrame:
    prompts = frame["prompt_bn"].tolist()

    char_query = char_vectorizer.transform(prompts)
    word_query = word_vectorizer.transform(prompts)

    char_distance, char_indices = char_nn.kneighbors(
        char_query,
        n_neighbors=TOP_CANDIDATES_PER_CHANNEL,
        return_distance=True,
    )

    word_distance, word_indices = word_nn.kneighbors(
        word_query,
        n_neighbors=TOP_CANDIDATES_PER_CHANNEL,
        return_distance=True,
    )

    char_similarity = 1.0 - char_distance
    word_similarity = 1.0 - word_distance

    output_rows: List[Dict[str, object]] = []

    iterator = range(len(frame))

    for row_position in tqdm(
        iterator,
        desc=f"Hybrid retrieval: {split_name}",
    ):
        candidate_scores: Dict[int, Dict[str, float]] = {}

        for rank, (kb_index, similarity) in enumerate(
            zip(
                char_indices[row_position],
                char_similarity[row_position],
            ),
            start=1,
        ):
            item = candidate_scores.setdefault(
                int(kb_index),
                {
                    "rrf": 0.0,
                    "char_similarity": 0.0,
                    "word_similarity": 0.0,
                    "char_rank": 999.0,
                    "word_rank": 999.0,
                },
            )
            item["rrf"] += 1.0 / (RRF_K + rank)
            item["char_similarity"] = max(
                item["char_similarity"],
                float(similarity),
            )
            item["char_rank"] = min(
                item["char_rank"],
                float(rank),
            )

        for rank, (kb_index, similarity) in enumerate(
            zip(
                word_indices[row_position],
                word_similarity[row_position],
            ),
            start=1,
        ):
            item = candidate_scores.setdefault(
                int(kb_index),
                {
                    "rrf": 0.0,
                    "char_similarity": 0.0,
                    "word_similarity": 0.0,
                    "char_rank": 999.0,
                    "word_rank": 999.0,
                },
            )
            item["rrf"] += 1.0 / (RRF_K + rank)
            item["word_similarity"] = max(
                item["word_similarity"],
                float(similarity),
            )
            item["word_rank"] = min(
                item["word_rank"],
                float(rank),
            )

        ranked = sorted(
            candidate_scores.items(),
            key=lambda pair: (
                pair[1]["rrf"],
                max(
                    pair[1]["char_similarity"],
                    pair[1]["word_similarity"],
                ),
            ),
            reverse=True,
        )

        # Keep unique evidence passages.
        selected: List[Tuple[int, Dict[str, float]]] = []
        seen_evidence: set[str] = set()

        for kb_index, score_data in ranked:
            evidence = clean_text(
                kb.iloc[kb_index]["kb_evidence"]
            )

            evidence_key = evidence.lower()

            if evidence_key in seen_evidence:
                continue

            seen_evidence.add(evidence_key)
            selected.append((kb_index, score_data))

            if len(selected) >= TOP_EVIDENCE:
                break

        for evidence_rank, (kb_index, score_data) in enumerate(
            selected,
            start=1,
        ):
            kb_row = kb.iloc[kb_index]

            output_rows.append({
                "row_position": int(row_position),
                "row_id": frame.iloc[row_position].get(
                    "id",
                    row_position,
                ),
                "evidence_rank": int(evidence_rank),
                "prompt_bn": frame.iloc[row_position]["prompt_bn"],
                "response_bn": frame.iloc[row_position]["response_bn"],
                "evidence": kb_row["kb_evidence"],
                "retrieved_prompt": kb_row["kb_prompt"],
                "retrieved_answer": kb_row["kb_answer"],
                "retrieved_route": kb_row["kb_route"],
                "retrieved_source": kb_row["kb_source"],
                "rrf_score": float(score_data["rrf"]),
                "char_similarity": float(
                    score_data["char_similarity"]
                ),
                "word_similarity": float(
                    score_data["word_similarity"]
                ),
                "max_lexical_similarity": float(
                    max(
                        score_data["char_similarity"],
                        score_data["word_similarity"],
                    )
                ),
                "exact_prompt_match": float(
                    clean_text(
                        frame.iloc[row_position]["prompt_bn"]
                    ).lower()
                    == clean_text(
                        kb_row["kb_prompt"]
                    ).lower()
                ),
            })

    pairs = pd.DataFrame(output_rows)

    expected_rows = len(frame) * TOP_EVIDENCE

    if len(pairs) != expected_rows:
        raise RuntimeError(
            f"Expected {expected_rows} retrieved pairs, got {len(pairs)}."
        )

    return pairs


# =============================================================================
# CONTEXT MODEL VERIFIER
# =============================================================================

class VerificationDataset(Dataset):
    def __init__(
        self,
        pairs: pd.DataFrame,
        tokenizer,
        max_length: int,
    ) -> None:
        self.evidence = pairs["evidence"].astype(str).tolist()
        self.prompts = pairs["prompt_bn"].astype(str).tolist()
        self.responses = pairs["response_bn"].astype(str).tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self) -> int:
        return len(self.evidence)

    def __getitem__(self, index: int) -> Dict[str, object]:
        second_sequence = (
            f"প্রশ্ন: {self.prompts[index]} "
            f"উত্তর: {self.responses[index]}"
        )

        return self.tokenizer(
            self.evidence[index],
            second_sequence,
            truncation="longest_first",
            max_length=self.max_length,
            padding=False,
        )


def make_verification_loader(
    pairs: pd.DataFrame,
    tokenizer,
) -> DataLoader:
    dataset = VerificationDataset(
        pairs=pairs,
        tokenizer=tokenizer,
        max_length=MAX_LENGTH,
    )

    collator = DataCollatorWithPadding(
        tokenizer=tokenizer,
        padding="longest",
        return_tensors="pt",
    )

    return DataLoader(
        dataset,
        batch_size=VERIFY_BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
        persistent_workers=(NUM_WORKERS > 0),
        collate_fn=collator,
    )


def autocast_context(device: torch.device):
    if device.type == "cuda":
        return torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
        )

    return nullcontext()


@torch.no_grad()
def predict_verifier(
    model,
    loader: DataLoader,
    device: torch.device,
    description: str,
) -> np.ndarray:
    model.eval()
    model.to(device)

    probabilities: List[np.ndarray] = []

    for batch in tqdm(
        loader,
        desc=description,
    ):
        batch = {
            key: value.to(
                device,
                non_blocking=True,
            )
            for key, value in batch.items()
        }

        with autocast_context(device):
            logits = model(**batch).logits

        probability = torch.softmax(
            logits.float(),
            dim=-1,
        )[:, 1]

        probabilities.append(
            probability.cpu().numpy()
        )

    return np.concatenate(probabilities)


def verify_pairs_with_context_models(
    oof_pairs: pd.DataFrame,
    test_pairs: pd.DataFrame,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    device = torch.device(
        "cuda"
        if torch.cuda.is_available()
        else "cpu"
    )

    print("\n[VERIFIER]")
    print("Device:", device)

    tokenizer = AutoTokenizer.from_pretrained(
        FOLD_MODEL_DIRS[0],
        use_fast=True,
    )

    combined = pd.concat(
        [
            oof_pairs.assign(_split="oof"),
            test_pairs.assign(_split="test"),
        ],
        ignore_index=True,
    )

    loader = make_verification_loader(
        pairs=combined,
        tokenizer=tokenizer,
    )

    fold_probabilities: List[np.ndarray] = []

    for fold, model_dir in enumerate(FOLD_MODEL_DIRS):
        print(f"\nLoading context verifier fold {fold}: {model_dir}")

        model = (
            AutoModelForSequenceClassification
            .from_pretrained(model_dir)
        )

        fold_probability = predict_verifier(
            model=model,
            loader=loader,
            device=device,
            description=f"Verifier fold {fold}",
        )

        fold_probabilities.append(
            fold_probability
        )

        del model
        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    probability_matrix = np.vstack(
        fold_probabilities
    )

    combined["verifier_prob_fair"] = (
        probability_matrix.mean(axis=0)
    )

    combined["verifier_fold_std"] = (
        probability_matrix.std(axis=0)
    )

    oof_verified = (
        combined[
            combined["_split"].eq("oof")
        ]
        .drop(columns=["_split"])
        .reset_index(drop=True)
    )

    test_verified = (
        combined[
            combined["_split"].eq("test")
        ]
        .drop(columns=["_split"])
        .reset_index(drop=True)
    )

    return oof_verified, test_verified


# =============================================================================
# ROW-LEVEL VERIFIER FEATURES
# =============================================================================

def aggregate_verifier_features(
    base_frame: pd.DataFrame,
    pairs: pd.DataFrame,
) -> pd.DataFrame:
    pairs = pairs.copy()

    # Normalize retrieval confidence per row before weighted averaging.
    pairs["retrieval_weight"] = (
        0.45 * pairs["char_similarity"]
        + 0.45 * pairs["word_similarity"]
        + 0.10 * (
            pairs["rrf_score"]
            / pairs.groupby("row_position")[
                "rrf_score"
            ].transform("max").clip(lower=1e-8)
        )
    ).clip(lower=1e-6)

    pairs["weighted_support_component"] = (
        pairs["verifier_prob_fair"]
        * pairs["retrieval_weight"]
    )

    grouped = pairs.groupby(
        "row_position",
        sort=True,
    )

    aggregate = grouped.agg(
        verifier_top1=(
            "verifier_prob_fair",
            "first",
        ),
        verifier_max=(
            "verifier_prob_fair",
            "max",
        ),
        verifier_mean=(
            "verifier_prob_fair",
            "mean",
        ),
        verifier_min=(
            "verifier_prob_fair",
            "min",
        ),
        verifier_std=(
            "verifier_prob_fair",
            "std",
        ),
        verifier_fold_std_mean=(
            "verifier_fold_std",
            "mean",
        ),
        retrieval_top1_char=(
            "char_similarity",
            "first",
        ),
        retrieval_top1_word=(
            "word_similarity",
            "first",
        ),
        retrieval_max_lexical=(
            "max_lexical_similarity",
            "max",
        ),
        retrieval_mean_lexical=(
            "max_lexical_similarity",
            "mean",
        ),
        exact_prompt_any=(
            "exact_prompt_match",
            "max",
        ),
    ).reset_index()

    weighted_support = (
        grouped["weighted_support_component"].sum()
        / grouped["retrieval_weight"].sum()
    ).rename("verifier_weighted")

    support_count_05 = grouped[
        "verifier_prob_fair"
    ].apply(
        lambda series: float(
            (series >= 0.50).sum()
        ) / TOP_EVIDENCE
    ).rename("verifier_support_fraction_05")

    support_count_08 = grouped[
        "verifier_prob_fair"
    ].apply(
        lambda series: float(
            (series >= 0.80).sum()
        ) / TOP_EVIDENCE
    ).rename("verifier_support_fraction_08")

    aggregate = aggregate.merge(
        weighted_support.reset_index(),
        on="row_position",
        how="left",
    )

    aggregate = aggregate.merge(
        support_count_05.reset_index(),
        on="row_position",
        how="left",
    )

    aggregate = aggregate.merge(
        support_count_08.reset_index(),
        on="row_position",
        how="left",
    )

    output = base_frame.copy().reset_index(drop=True)
    output["row_position"] = np.arange(
        len(output),
        dtype=np.int64,
    )

    output = output.merge(
        aggregate,
        on="row_position",
        how="left",
        validate="one_to_one",
    )

    # Keep top-1 evidence for inspection.
    top1 = (
        pairs[
            pairs["evidence_rank"].eq(1)
        ][
            [
                "row_position",
                "evidence",
                "retrieved_prompt",
                "retrieved_answer",
                "retrieved_route",
                "retrieved_source",
            ]
        ]
        .rename(
            columns={
                "evidence": "step6_top1_evidence",
                "retrieved_prompt": "step6_top1_prompt",
                "retrieved_answer": "step6_top1_answer",
                "retrieved_route": "step6_top1_route",
                "retrieved_source": "step6_top1_source",
            }
        )
    )

    output = output.merge(
        top1,
        on="row_position",
        how="left",
        validate="one_to_one",
    )

    output["verifier_gap"] = (
        output["verifier_max"]
        - output["verifier_min"]
    )

    output["verifier_retrieval_joint"] = (
        output["verifier_weighted"]
        * output["retrieval_max_lexical"]
    )

    return output


# =============================================================================
# META MODEL
# =============================================================================

META_FEATURES = [
    "retrieval_ensemble_prob_fair",
    "prob_fair",
    "prob_std",
    "retr_prompt_top1",
    "retr_answer_max",
    "retr_answer_weighted",
    "retr_joint_max",
    "verifier_top1",
    "verifier_max",
    "verifier_mean",
    "verifier_min",
    "verifier_std",
    "verifier_fold_std_mean",
    "verifier_weighted",
    "verifier_support_fraction_05",
    "verifier_support_fraction_08",
    "retrieval_top1_char",
    "retrieval_top1_word",
    "retrieval_max_lexical",
    "retrieval_mean_lexical",
    "exact_prompt_any",
    "verifier_gap",
    "verifier_retrieval_joint",
    "response_len",
    "prompt_len",
]


def feature_matrix(
    frame: pd.DataFrame,
    features: Sequence[str],
) -> np.ndarray:
    return (
        frame[list(features)]
        .apply(pd.to_numeric, errors="coerce")
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0.0)
        .to_numpy(dtype=np.float32)
    )


def tune_blend(
    y_true: np.ndarray,
    meta_probability: np.ndarray,
    baseline_probability: np.ndarray,
) -> Dict[str, object]:
    best: Optional[Dict[str, object]] = None
    best_key: Optional[Tuple[float, ...]] = None

    for meta_weight in BLEND_WEIGHTS:
        probability = (
            meta_weight * meta_probability
            + (1.0 - meta_weight) * baseline_probability
        )

        metrics = tune_threshold(
            y_true=y_true,
            probability=probability,
        )

        candidate = {
            "meta_weight": float(meta_weight),
            "baseline_weight": float(
                1.0 - meta_weight
            ),
            **metrics,
        }

        key = (
            float(candidate["macro_f1"]),
            float(candidate["accuracy"]),
            float(candidate["f1_label0"]),
            float(candidate["f1_label1"]),
        )

        if best is None or key > best_key:
            best = candidate
            best_key = key

    assert best is not None
    return best


def train_meta_model(
    oof_features: pd.DataFrame,
    test_features: pd.DataFrame,
):
    available_features = [
        feature
        for feature in META_FEATURES
        if feature in oof_features.columns
        and feature in test_features.columns
    ]

    print("\n[META MODEL]")
    print("Feature count:", len(available_features))

    for feature in available_features:
        print(" -", feature)

    X = feature_matrix(
        oof_features,
        available_features,
    )

    X_test = feature_matrix(
        test_features,
        available_features,
    )

    y = oof_features["label"].astype(int).to_numpy()

    baseline_oof = pd.to_numeric(
        oof_features[
            "retrieval_ensemble_prob_fair"
        ],
        errors="coerce",
    ).fillna(0.5).to_numpy(dtype=np.float64)

    baseline_test = pd.to_numeric(
        test_features[
            "retrieval_ensemble_prob_fair"
        ],
        errors="coerce",
    ).fillna(0.5).to_numpy(dtype=np.float64)

    splitter = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=SEED,
    )

    baseline_metrics = tune_threshold(
        y_true=y,
        probability=baseline_oof,
    )

    candidates: List[Dict[str, object]] = []
    meta_oof_by_c: Dict[float, np.ndarray] = {}

    for c_value in META_C_VALUES:
        model = LogisticRegression(
            C=float(c_value),
            class_weight="balanced",
            max_iter=4000,
            solver="liblinear",
            random_state=SEED,
        )

        meta_oof = cross_val_predict(
            model,
            X,
            y,
            cv=splitter,
            method="predict_proba",
            n_jobs=-1,
        )[:, 1]

        meta_oof_by_c[float(c_value)] = meta_oof

        pure_metrics = tune_threshold(
            y_true=y,
            probability=meta_oof,
        )

        blend_metrics = tune_blend(
            y_true=y,
            meta_probability=meta_oof,
            baseline_probability=baseline_oof,
        )

        candidates.append({
            "C": float(c_value),
            "mode": "meta_only",
            "meta_weight": 1.0,
            "baseline_weight": 0.0,
            **pure_metrics,
        })

        candidates.append({
            "C": float(c_value),
            "mode": "blend",
            **blend_metrics,
        })

        print(
            f"C={c_value} | "
            f"meta Macro-F1={pure_metrics['macro_f1']:.6f} | "
            f"blend Macro-F1={blend_metrics['macro_f1']:.6f}"
        )

    selected = max(
        candidates,
        key=lambda item: (
            float(item["macro_f1"]),
            float(item["accuracy"]),
            float(item["f1_label0"]),
            float(item["f1_label1"]),
        ),
    )

    selected_c = float(selected["C"])
    selected_meta_oof = meta_oof_by_c[
        selected_c
    ]

    final_model = LogisticRegression(
        C=selected_c,
        class_weight="balanced",
        max_iter=4000,
        solver="liblinear",
        random_state=SEED,
    )

    final_model.fit(X, y)

    meta_test = final_model.predict_proba(
        X_test
    )[:, 1]

    selected_oof_probability = (
        float(selected["meta_weight"])
        * selected_meta_oof
        + float(selected["baseline_weight"])
        * baseline_oof
    )

    selected_test_probability = (
        float(selected["meta_weight"])
        * meta_test
        + float(selected["baseline_weight"])
        * baseline_test
    )

    improvement = (
        float(selected["macro_f1"])
        - float(baseline_metrics["macro_f1"])
    )

    use_step6 = bool(improvement > 0.0)

    if use_step6:
        final_oof_probability = selected_oof_probability
        final_test_probability = selected_test_probability
        final_threshold = float(selected["threshold"])
        final_method = (
            f"step6_{selected['mode']}_C{selected_c}"
        )
    else:
        final_oof_probability = baseline_oof
        final_test_probability = baseline_test
        final_threshold = float(
            baseline_metrics["threshold"]
        )
        final_method = "step3_retrieval_fallback"

    return {
        "features": available_features,
        "baseline_metrics": baseline_metrics,
        "selected": selected,
        "improvement": float(improvement),
        "use_step6": use_step6,
        "final_method": final_method,
        "final_threshold": float(final_threshold),
        "final_model": final_model,
        "meta_oof_probability": selected_meta_oof,
        "meta_test_probability": meta_test,
        "final_oof_probability": final_oof_probability,
        "final_test_probability": final_test_probability,
    }


# =============================================================================
# MAIN
# =============================================================================

def main() -> None:
    seed_everything(SEED)

    OUTPUT_DIR.mkdir(
        parents=True,
        exist_ok=True,
    )

    print("=" * 96)
    print("STEP 6 — HYBRID RETRIEVAL + TOP-5 CONTEXT VERIFIER")
    print("=" * 96)

    pool, oof, test = load_inputs()

    print("\n[INPUT]")
    print("Official null OOF rows:", len(oof))
    print("Null test rows:", len(test))

    kb = build_knowledge_base(pool)

    (
        char_vectorizer,
        char_nn,
        word_vectorizer,
        word_nn,
    ) = build_retrievers(kb)

    print("\nRetrieving top-5 evidence for official null OOF...")

    oof_pairs = retrieve_hybrid(
        frame=oof,
        kb=kb,
        char_vectorizer=char_vectorizer,
        char_nn=char_nn,
        word_vectorizer=word_vectorizer,
        word_nn=word_nn,
        split_name="OOF",
    )

    print("\nRetrieving top-5 evidence for null test...")

    test_pairs = retrieve_hybrid(
        frame=test,
        kb=kb,
        char_vectorizer=char_vectorizer,
        char_nn=char_nn,
        word_vectorizer=word_vectorizer,
        word_nn=word_nn,
        split_name="TEST",
    )

    # Free large sparse matrices before transformer inference.
    del (
        char_nn,
        word_nn,
        char_vectorizer,
        word_vectorizer,
    )
    gc.collect()

    (
        oof_pairs,
        test_pairs,
    ) = verify_pairs_with_context_models(
        oof_pairs=oof_pairs,
        test_pairs=test_pairs,
    )

    oof_pairs.to_csv(
        OUTPUT_DIR
        / "step6_retrieved_pairs_oof.csv",
        index=False,
    )

    test_pairs.to_csv(
        OUTPUT_DIR
        / "step6_retrieved_pairs_test.csv",
        index=False,
    )

    print("\nAggregating top-5 verifier features...")

    oof_features = aggregate_verifier_features(
        base_frame=oof,
        pairs=oof_pairs,
    )

    test_features = aggregate_verifier_features(
        base_frame=test,
        pairs=test_pairs,
    )

    result = train_meta_model(
        oof_features=oof_features,
        test_features=test_features,
    )

    final_threshold = float(
        result["final_threshold"]
    )

    oof_features[
        "step6_meta_prob_fair"
    ] = result["meta_oof_probability"]

    oof_features[
        "step6_final_prob_fair"
    ] = result["final_oof_probability"]

    oof_features[
        "step6_final_pred"
    ] = (
        result["final_oof_probability"]
        >= final_threshold
    ).astype(int)

    test_features[
        "step6_meta_prob_fair"
    ] = result["meta_test_probability"]

    test_features[
        "step6_final_prob_fair"
    ] = result["final_test_probability"]

    test_features[
        "step6_final_pred"
    ] = (
        result["final_test_probability"]
        >= final_threshold
    ).astype(int)

    oof_features.to_csv(
        OUTPUT_DIR / "step6_null_oof.csv",
        index=False,
    )

    test_features.to_csv(
        OUTPUT_DIR / "step6_null_test.csv",
        index=False,
    )

    report = {
        "knowledge_base_rows": int(len(kb)),
        "knowledge_base_route_counts": {
            str(key): int(value)
            for key, value in kb[
                "kb_route"
            ].value_counts().to_dict().items()
        },
        "top_candidates_per_channel": int(
            TOP_CANDIDATES_PER_CHANNEL
        ),
        "top_evidence": int(TOP_EVIDENCE),
        "meta_features": result["features"],
        "baseline_metrics": result[
            "baseline_metrics"
        ],
        "selected": result["selected"],
        "macro_f1_improvement": result[
            "improvement"
        ],
        "use_step6": result["use_step6"],
        "final_method": result[
            "final_method"
        ],
        "final_threshold": result[
            "final_threshold"
        ],
        "test_prediction_counts": {
            str(key): int(value)
            for key, value in test_features[
                "step6_final_pred"
            ].value_counts().to_dict().items()
        },
    }

    with open(
        OUTPUT_DIR / "step6_metrics.json",
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            report,
            file,
            ensure_ascii=False,
            indent=2,
        )

    print("\n" + "=" * 96)
    print("STEP 6 RESULT")
    print("=" * 96)

    print(json.dumps(
        report,
        ensure_ascii=False,
        indent=2,
    ))

    print("\nSaved:")
    print(OUTPUT_DIR / "step6_null_oof.csv")
    print(OUTPUT_DIR / "step6_null_test.csv")
    print(OUTPUT_DIR / "step6_metrics.json")
    print(OUTPUT_DIR / "step6_retrieved_pairs_oof.csv")
    print(OUTPUT_DIR / "step6_retrieved_pairs_test.csv")


if __name__ == "__main__":
    main()


## 4. Verify every old-null output


In [ ]:
required_stage_outputs = [
    Path("/kaggle/working/null_model_step2/null_oof_predictions.csv"),
    Path("/kaggle/working/null_model_step2/null_test_predictions.csv"),
    Path("/kaggle/working/null_retrieval_step3/null_retrieval_oof.csv"),
    Path("/kaggle/working/null_retrieval_step3/null_retrieval_test.csv"),
    Path("/kaggle/working/context_model_step4/context_oof_predictions.csv"),
    Path("/kaggle/working/context_model_step4/context_test_predictions.csv"),
    Path("/kaggle/working/null_step6_hybrid_verifier/step6_null_oof.csv"),
    Path("/kaggle/working/null_step6_hybrid_verifier/step6_null_test.csv"),
]

print("=" * 88)
print("OLD 0.732 NULL PIPELINE OUTPUT CHECK")
print("=" * 88)

for path in required_stage_outputs:
    if not path.exists():
        raise FileNotFoundError(path)
    print("READY:", path)


## 5. Cross-validated route-wise fusion

This stage creates two independent fusion candidates for each route:

1. **Nested Logistic Stacking:** learns how to combine base probabilities using only fold-safe OOF signals.
2. **Nested Soft Voting:** selects probability weights and a decision threshold inside each outer fold.

The original 0.745 hybrid system remains the anchor. Fusion is used in the safe submission only when it improves label-0 F1 without a material Macro-F1 loss.


In [ ]:

from __future__ import annotations

import json
import math
from itertools import product
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


# =============================================================================
# FUSION CONFIGURATION
# =============================================================================

FUSION_SEED = 20260717
FUSION_INNER_SPLITS = 4
FUSION_THRESHOLDS = np.arange(0.15, 0.851, 0.01)

# A fused route is accepted only when it improves the main hallucinated-class
# F1 and does not materially damage Macro-F1.
MIN_LABEL0_GAIN = 0.002
MAX_MACRO_DROP = 0.002

FUSION_DIR = Path("/kaggle/working/hybrid_cv_fusion")
FUSION_DIR.mkdir(parents=True, exist_ok=True)


# =============================================================================
# METRICS AND ALIGNMENT
# =============================================================================

def fusion_metrics(y_true, prediction):
    y_true = np.asarray(y_true, dtype=np.int64)
    prediction = np.asarray(prediction, dtype=np.int64)

    return {
        "accuracy": float(accuracy_score(y_true, prediction)),
        "f1_label0": float(
            f1_score(y_true, prediction, pos_label=0, zero_division=0)
        ),
        "f1_label1": float(
            f1_score(y_true, prediction, pos_label=1, zero_division=0)
        ),
        "macro_f1": float(
            f1_score(y_true, prediction, average="macro", zero_division=0)
        ),
        "confusion_matrix": confusion_matrix(
            y_true,
            prediction,
            labels=[0, 1],
        ).tolist(),
    }


def metric_key(metric):
    return (
        metric["f1_label0"],
        metric["macro_f1"],
        metric["accuracy"],
        metric["f1_label1"],
    )


def tune_fusion_threshold(y_true, probability):
    probability = np.asarray(probability, dtype=np.float64)
    best = None
    best_key = None

    for threshold in FUSION_THRESHOLDS:
        prediction = (probability >= threshold).astype(np.int64)
        current = {
            "threshold": float(threshold),
            **fusion_metrics(y_true, prediction),
        }
        current_key = metric_key(current)

        if best is None or current_key > best_key:
            best = current
            best_key = current_key

    return best


def prepare_key(frame):
    output = frame.copy()
    output["_fusion_id"] = output["id"].astype(str).str.strip()
    return output


def merge_signal(base, signal, rename_map, signal_name):
    left = prepare_key(base)
    right = prepare_key(signal)

    missing = [
        column
        for column in rename_map
        if column not in right.columns
    ]
    if missing:
        raise ValueError(
            f"{signal_name} is missing columns {missing}. "
            f"Available: {right.columns.tolist()}"
        )

    right = right[
        ["_fusion_id"] + list(rename_map)
    ].drop_duplicates("_fusion_id", keep="last")

    right = right.rename(columns=rename_map)

    merged = left.merge(
        right,
        on="_fusion_id",
        how="left",
        validate="one_to_one",
    )

    new_columns = list(rename_map.values())
    if merged[new_columns].isna().any().any():
        missing_ids = merged.loc[
            merged[new_columns].isna().any(axis=1),
            "id",
        ].tolist()[:10]
        raise ValueError(
            f"{signal_name} alignment failed. Missing IDs: {missing_ids}"
        )

    return merged.drop(columns=["_fusion_id"])


# =============================================================================
# FUSION FEATURES
# =============================================================================

def build_probability_features(frame, probability_columns):
    probabilities = (
        frame[probability_columns]
        .apply(pd.to_numeric, errors="raise")
        .astype(float)
        .clip(1e-5, 1.0 - 1e-5)
    )

    features = probabilities.copy()
    values = probabilities.to_numpy(dtype=np.float64)

    features["fusion_mean"] = values.mean(axis=1)
    features["fusion_std"] = values.std(axis=1)
    features["fusion_min"] = values.min(axis=1)
    features["fusion_max"] = values.max(axis=1)
    features["fusion_range"] = (
        features["fusion_max"] - features["fusion_min"]
    )
    features["fusion_product"] = values.prod(axis=1)
    features["fusion_geometric_mean"] = np.exp(
        np.log(values).mean(axis=1)
    )

    for column in probability_columns:
        features[f"{column}_confidence"] = (
            np.abs(probabilities[column] - 0.5) * 2.0
        )

    for first_index in range(len(probability_columns)):
        for second_index in range(first_index + 1, len(probability_columns)):
            first = probability_columns[first_index]
            second = probability_columns[second_index]
            features[f"absdiff_{first}_{second}"] = np.abs(
                probabilities[first] - probabilities[second]
            )
            features[f"product_{first}_{second}"] = (
                probabilities[first] * probabilities[second]
            )

    return (
        features
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0.0)
    )


def make_fusion_model(c_value, random_state):
    return Pipeline(
        [
            ("scale", StandardScaler()),
            (
                "model",
                LogisticRegression(
                    C=float(c_value),
                    class_weight="balanced",
                    solver="liblinear",
                    max_iter=5000,
                    random_state=int(random_state),
                ),
            ),
        ]
    )


# =============================================================================
# NESTED LOGISTIC STACKING
# =============================================================================

def nested_logistic_fusion(
    X_train,
    y,
    folds,
    X_test,
    route_name,
):
    X_train = np.asarray(X_train, dtype=np.float64)
    X_test = np.asarray(X_test, dtype=np.float64)
    y = np.asarray(y, dtype=np.int64)
    folds = np.asarray(folds, dtype=np.int64)

    c_grid = [0.03, 0.05, 0.10, 0.25, 0.50, 1.00, 2.00]

    outer_probability = np.full(len(y), np.nan, dtype=np.float64)
    outer_prediction = np.full(len(y), -1, dtype=np.int64)

    test_fold_probability = []
    outer_settings = []

    for outer_fold in sorted(np.unique(folds)):
        outer_train_index = np.where(folds != outer_fold)[0]
        outer_valid_index = np.where(folds == outer_fold)[0]

        X_outer_train = X_train[outer_train_index]
        y_outer_train = y[outer_train_index]

        minimum_class = int(np.bincount(y_outer_train).min())
        inner_splits = min(FUSION_INNER_SPLITS, minimum_class)

        if inner_splits < 2:
            raise ValueError(
                f"{route_name}: not enough samples for inner CV."
            )

        inner_cv = StratifiedKFold(
            n_splits=inner_splits,
            shuffle=True,
            random_state=FUSION_SEED + int(outer_fold),
        )

        best_setting = None
        best_setting_key = None

        for c_value in c_grid:
            inner_probability = np.full(
                len(outer_train_index),
                np.nan,
                dtype=np.float64,
            )

            for inner_fold, (
                inner_train_local,
                inner_valid_local,
            ) in enumerate(
                inner_cv.split(X_outer_train, y_outer_train)
            ):
                model = make_fusion_model(
                    c_value,
                    FUSION_SEED
                    + int(outer_fold) * 100
                    + int(inner_fold),
                )
                model.fit(
                    X_outer_train[inner_train_local],
                    y_outer_train[inner_train_local],
                )
                inner_probability[inner_valid_local] = (
                    model.predict_proba(
                        X_outer_train[inner_valid_local]
                    )[:, 1]
                )

            threshold_result = tune_fusion_threshold(
                y_outer_train,
                inner_probability,
            )

            setting_key = metric_key(threshold_result)

            if (
                best_setting is None
                or setting_key > best_setting_key
            ):
                best_setting = {
                    "C": float(c_value),
                    "threshold": float(
                        threshold_result["threshold"]
                    ),
                    "inner_metrics": threshold_result,
                }
                best_setting_key = setting_key

        final_model = make_fusion_model(
            best_setting["C"],
            FUSION_SEED + 1000 + int(outer_fold),
        )
        final_model.fit(X_outer_train, y_outer_train)

        valid_probability = final_model.predict_proba(
            X_train[outer_valid_index]
        )[:, 1]

        valid_prediction = (
            valid_probability >= best_setting["threshold"]
        ).astype(np.int64)

        outer_probability[outer_valid_index] = valid_probability
        outer_prediction[outer_valid_index] = valid_prediction

        test_fold_probability.append(
            final_model.predict_proba(X_test)[:, 1]
        )

        joblib.dump(
            final_model,
            FUSION_DIR
            / f"{route_name}_logistic_outer_fold_{outer_fold}.joblib",
        )

        outer_settings.append(
            {
                "outer_fold": int(outer_fold),
                "C": float(best_setting["C"]),
                "threshold": float(best_setting["threshold"]),
                "valid_rows": int(len(outer_valid_index)),
                "valid_metrics": fusion_metrics(
                    y[outer_valid_index],
                    valid_prediction,
                ),
                "inner_metrics": best_setting["inner_metrics"],
            }
        )

    if np.isnan(outer_probability).any():
        raise RuntimeError(
            f"{route_name}: incomplete nested stacking OOF."
        )

    test_probability = np.vstack(
        test_fold_probability
    ).mean(axis=0)

    final_threshold = float(
        np.median(
            [setting["threshold"] for setting in outer_settings]
        )
    )

    return {
        "name": "nested_logistic_stacking",
        "oof_probability": outer_probability,
        "oof_prediction": outer_prediction,
        "test_probability": test_probability,
        "test_prediction": (
            test_probability >= final_threshold
        ).astype(np.int64),
        "final_threshold": final_threshold,
        "metrics": fusion_metrics(y, outer_prediction),
        "outer_settings": outer_settings,
    }


# =============================================================================
# NESTED SOFT-VOTING FUSION
# =============================================================================

def candidate_weight_vectors(n_probabilities):
    if n_probabilities == 2:
        return [
            np.asarray([weight, 1.0 - weight], dtype=np.float64)
            for weight in np.arange(0.0, 1.001, 0.10)
        ]

    if n_probabilities == 4:
        raw = [
            [0.00, 0.00, 0.00, 1.00],
            [0.10, 0.10, 0.10, 0.70],
            [0.10, 0.15, 0.20, 0.55],
            [0.10, 0.20, 0.20, 0.50],
            [0.15, 0.20, 0.20, 0.45],
            [0.15, 0.25, 0.20, 0.40],
            [0.20, 0.20, 0.20, 0.40],
            [0.20, 0.25, 0.20, 0.35],
            [0.25, 0.25, 0.25, 0.25],
            [0.30, 0.25, 0.15, 0.30],
            [0.35, 0.30, 0.10, 0.25],
            [0.40, 0.30, 0.10, 0.20],
        ]
        return [
            np.asarray(weight, dtype=np.float64)
            / np.sum(weight)
            for weight in raw
        ]

    raise ValueError(
        f"Unsupported number of probabilities: {n_probabilities}"
    )


def nested_soft_vote(
    train_probabilities,
    y,
    folds,
    test_probabilities,
    route_name,
):
    train_probabilities = np.asarray(
        train_probabilities,
        dtype=np.float64,
    )
    test_probabilities = np.asarray(
        test_probabilities,
        dtype=np.float64,
    )
    y = np.asarray(y, dtype=np.int64)
    folds = np.asarray(folds, dtype=np.int64)

    weights = candidate_weight_vectors(
        train_probabilities.shape[1]
    )

    outer_probability = np.full(len(y), np.nan, dtype=np.float64)
    outer_prediction = np.full(len(y), -1, dtype=np.int64)

    selected_weights = []
    selected_thresholds = []
    outer_settings = []

    for outer_fold in sorted(np.unique(folds)):
        train_index = np.where(folds != outer_fold)[0]
        valid_index = np.where(folds == outer_fold)[0]

        best = None
        best_key = None

        for current_weights in weights:
            train_blend = (
                train_probabilities[train_index]
                @ current_weights
            )
            threshold_result = tune_fusion_threshold(
                y[train_index],
                train_blend,
            )

            current_key = metric_key(threshold_result)

            if best is None or current_key > best_key:
                best = {
                    "weights": current_weights,
                    "threshold": float(
                        threshold_result["threshold"]
                    ),
                    "train_metrics": threshold_result,
                }
                best_key = current_key

        valid_probability = (
            train_probabilities[valid_index]
            @ best["weights"]
        )
        valid_prediction = (
            valid_probability >= best["threshold"]
        ).astype(np.int64)

        outer_probability[valid_index] = valid_probability
        outer_prediction[valid_index] = valid_prediction

        selected_weights.append(best["weights"])
        selected_thresholds.append(best["threshold"])

        outer_settings.append(
            {
                "outer_fold": int(outer_fold),
                "weights": [
                    float(value)
                    for value in best["weights"]
                ],
                "threshold": float(best["threshold"]),
                "valid_rows": int(len(valid_index)),
                "valid_metrics": fusion_metrics(
                    y[valid_index],
                    valid_prediction,
                ),
                "train_metrics": best["train_metrics"],
            }
        )

    final_weights = np.mean(
        np.vstack(selected_weights),
        axis=0,
    )
    final_weights = final_weights / final_weights.sum()

    final_threshold = float(np.median(selected_thresholds))

    test_probability = (
        test_probabilities @ final_weights
    )
    test_prediction = (
        test_probability >= final_threshold
    ).astype(np.int64)

    return {
        "name": "nested_soft_vote",
        "oof_probability": outer_probability,
        "oof_prediction": outer_prediction,
        "test_probability": test_probability,
        "test_prediction": test_prediction,
        "final_threshold": final_threshold,
        "final_weights": [
            float(value)
            for value in final_weights
        ],
        "metrics": fusion_metrics(y, outer_prediction),
        "outer_settings": outer_settings,
    }


# =============================================================================
# ROUTE CANDIDATE SELECTION
# =============================================================================

def select_route_fusion(
    route_name,
    anchor_probability,
    anchor_prediction,
    logistic_result,
    soft_vote_result,
    y,
):
    anchor_metrics = fusion_metrics(y, anchor_prediction)

    candidates = {
        "anchor": {
            "name": "anchor",
            "oof_probability": anchor_probability,
            "oof_prediction": anchor_prediction,
            "metrics": anchor_metrics,
        },
        "nested_logistic_stacking": logistic_result,
        "nested_soft_vote": soft_vote_result,
    }

    best_fusion_name = max(
        ["nested_logistic_stacking", "nested_soft_vote"],
        key=lambda name: metric_key(
            candidates[name]["metrics"]
        ),
    )
    best_fusion = candidates[best_fusion_name]

    label0_gain = (
        best_fusion["metrics"]["f1_label0"]
        - anchor_metrics["f1_label0"]
    )
    macro_gain = (
        best_fusion["metrics"]["macro_f1"]
        - anchor_metrics["macro_f1"]
    )

    accepted = bool(
        label0_gain >= MIN_LABEL0_GAIN
        and macro_gain >= -MAX_MACRO_DROP
    )

    safe_name = (
        best_fusion_name
        if accepted
        else "anchor"
    )

    return {
        "route": route_name,
        "anchor_metrics": anchor_metrics,
        "candidate_metrics": {
            name: candidate["metrics"]
            for name, candidate in candidates.items()
        },
        "best_fusion_name": best_fusion_name,
        "label0_gain": float(label0_gain),
        "macro_gain": float(macro_gain),
        "accepted": accepted,
        "safe_name": safe_name,
        "candidates": candidates,
    }


# =============================================================================
# LOAD OFFICIAL AND TEST DATA
# =============================================================================

official = normalize_frame(
    pd.read_csv(
        PREPROCESSED_DIR / "official_folds.csv",
        keep_default_na=False,
    ),
    True,
)
competition_test = normalize_frame(
    pd.read_csv(
        PREPROCESSED_DIR / "competition_test.csv",
        keep_default_na=False,
    ),
    False,
)

official["fold"] = pd.to_numeric(
    official["fold"],
    errors="raise",
).astype(int)

official_context = (
    official[official["route"].eq("context")]
    .copy()
    .reset_index(drop=True)
)
official_null = (
    official[official["route"].eq("null")]
    .copy()
    .reset_index(drop=True)
)
test_context = (
    competition_test[
        competition_test["route"].eq("context")
    ]
    .copy()
    .reset_index(drop=True)
)
test_null = (
    competition_test[
        competition_test["route"].eq("null")
    ]
    .copy()
    .reset_index(drop=True)
)


# =============================================================================
# LOAD BASE MODEL SIGNALS
# =============================================================================

context_lexical_oof = pd.read_csv(
    REPO_CONTEXT_DIR / "context_oof.csv",
    keep_default_na=False,
)
context_lexical_test = pd.read_csv(
    REPO_CONTEXT_DIR / "context_test.csv",
    keep_default_na=False,
)

context_neural_oof = pd.read_csv(
    "/kaggle/working/context_model_step4/"
    "context_oof_predictions.csv",
    keep_default_na=False,
)
context_neural_test = pd.read_csv(
    "/kaggle/working/context_model_step4/"
    "context_test_predictions.csv",
    keep_default_na=False,
)

null_direct_oof = pd.read_csv(
    "/kaggle/working/null_model_step2/"
    "null_oof_predictions.csv",
    keep_default_na=False,
)
null_direct_test = pd.read_csv(
    "/kaggle/working/null_model_step2/"
    "null_test_predictions.csv",
    keep_default_na=False,
)

null_retrieval_oof = pd.read_csv(
    "/kaggle/working/null_retrieval_step3/"
    "null_retrieval_oof.csv",
    keep_default_na=False,
)
null_retrieval_test = pd.read_csv(
    "/kaggle/working/null_retrieval_step3/"
    "null_retrieval_test.csv",
    keep_default_na=False,
)

null_step6_oof = pd.read_csv(
    "/kaggle/working/null_step6_hybrid_verifier/"
    "step6_null_oof.csv",
    keep_default_na=False,
)
null_step6_test = pd.read_csv(
    "/kaggle/working/null_step6_hybrid_verifier/"
    "step6_null_test.csv",
    keep_default_na=False,
)


# =============================================================================
# CONTEXT ROUTE SIGNAL TABLE
# =============================================================================

context_train_signals = merge_signal(
    official_context,
    context_lexical_oof,
    {
        "context_probability_fair": "context_lexical_prob",
        "context_prediction": "context_anchor_pred",
    },
    "context lexical OOF",
)
context_train_signals = merge_signal(
    context_train_signals,
    context_neural_oof,
    {
        "prob_fair": "context_neural_prob",
    },
    "context neural OOF",
)

context_test_signals = merge_signal(
    test_context,
    context_lexical_test,
    {
        "context_probability_fair": "context_lexical_prob",
        "context_prediction": "context_anchor_pred",
    },
    "context lexical test",
)
context_test_signals = merge_signal(
    context_test_signals,
    context_neural_test,
    {
        "prob_fair": "context_neural_prob",
    },
    "context neural test",
)

context_probability_columns = [
    "context_lexical_prob",
    "context_neural_prob",
]

context_X_train = build_probability_features(
    context_train_signals,
    context_probability_columns,
)
context_X_test = build_probability_features(
    context_test_signals,
    context_probability_columns,
)

context_y = context_train_signals[
    "label"
].to_numpy(dtype=np.int64)
context_folds = context_train_signals[
    "fold"
].to_numpy(dtype=np.int64)

context_logistic = nested_logistic_fusion(
    X_train=context_X_train.to_numpy(dtype=np.float64),
    y=context_y,
    folds=context_folds,
    X_test=context_X_test.to_numpy(dtype=np.float64),
    route_name="context",
)

context_soft_vote = nested_soft_vote(
    train_probabilities=context_train_signals[
        context_probability_columns
    ].to_numpy(dtype=np.float64),
    y=context_y,
    folds=context_folds,
    test_probabilities=context_test_signals[
        context_probability_columns
    ].to_numpy(dtype=np.float64),
    route_name="context",
)

context_selection = select_route_fusion(
    route_name="context",
    anchor_probability=context_train_signals[
        "context_lexical_prob"
    ].to_numpy(dtype=np.float64),
    anchor_prediction=context_train_signals[
        "context_anchor_pred"
    ].to_numpy(dtype=np.int64),
    logistic_result=context_logistic,
    soft_vote_result=context_soft_vote,
    y=context_y,
)


# =============================================================================
# NULL ROUTE SIGNAL TABLE
# =============================================================================

null_train_signals = merge_signal(
    official_null,
    null_direct_oof,
    {
        "prob_fair": "null_direct_prob",
    },
    "null direct OOF",
)
null_train_signals = merge_signal(
    null_train_signals,
    null_retrieval_oof,
    {
        "retrieval_ensemble_prob_fair": "null_retrieval_prob",
    },
    "null retrieval OOF",
)
null_train_signals = merge_signal(
    null_train_signals,
    null_step6_oof,
    {
        "step6_meta_prob_fair": "null_step6_meta_prob",
        "step6_final_prob_fair": "null_step6_prob",
        "step6_final_pred": "null_anchor_pred",
    },
    "null Step-6 OOF",
)

null_test_signals = merge_signal(
    test_null,
    null_direct_test,
    {
        "prob_fair": "null_direct_prob",
    },
    "null direct test",
)
null_test_signals = merge_signal(
    null_test_signals,
    null_retrieval_test,
    {
        "retrieval_ensemble_prob_fair": "null_retrieval_prob",
    },
    "null retrieval test",
)
null_test_signals = merge_signal(
    null_test_signals,
    null_step6_test,
    {
        "step6_meta_prob_fair": "null_step6_meta_prob",
        "step6_final_prob_fair": "null_step6_prob",
        "step6_final_pred": "null_anchor_pred",
    },
    "null Step-6 test",
)

null_probability_columns = [
    "null_direct_prob",
    "null_retrieval_prob",
    "null_step6_meta_prob",
    "null_step6_prob",
]

null_X_train = build_probability_features(
    null_train_signals,
    null_probability_columns,
)
null_X_test = build_probability_features(
    null_test_signals,
    null_probability_columns,
)

null_y = null_train_signals[
    "label"
].to_numpy(dtype=np.int64)
null_folds = null_train_signals[
    "fold"
].to_numpy(dtype=np.int64)

null_logistic = nested_logistic_fusion(
    X_train=null_X_train.to_numpy(dtype=np.float64),
    y=null_y,
    folds=null_folds,
    X_test=null_X_test.to_numpy(dtype=np.float64),
    route_name="null",
)

null_soft_vote = nested_soft_vote(
    train_probabilities=null_train_signals[
        null_probability_columns
    ].to_numpy(dtype=np.float64),
    y=null_y,
    folds=null_folds,
    test_probabilities=null_test_signals[
        null_probability_columns
    ].to_numpy(dtype=np.float64),
    route_name="null",
)

null_selection = select_route_fusion(
    route_name="null",
    anchor_probability=null_train_signals[
        "null_step6_prob"
    ].to_numpy(dtype=np.float64),
    anchor_prediction=null_train_signals[
        "null_anchor_pred"
    ].to_numpy(dtype=np.int64),
    logistic_result=null_logistic,
    soft_vote_result=null_soft_vote,
    y=null_y,
)


# =============================================================================
# EXTRACT FORCED AND SAFE ROUTE PREDICTIONS
# =============================================================================

def extract_route_output(
    selection,
    test_signals,
    anchor_probability_column,
    anchor_prediction_column,
):
    best_fusion = selection["candidates"][
        selection["best_fusion_name"]
    ]

    forced = {
        "name": selection["best_fusion_name"],
        "oof_probability": best_fusion["oof_probability"],
        "oof_prediction": best_fusion["oof_prediction"],
        "test_probability": best_fusion["test_probability"],
        "test_prediction": best_fusion["test_prediction"],
    }

    if selection["accepted"]:
        safe = forced
    else:
        safe = {
            "name": "anchor",
            "oof_probability": selection["candidates"][
                "anchor"
            ]["oof_probability"],
            "oof_prediction": selection["candidates"][
                "anchor"
            ]["oof_prediction"],
            "test_probability": test_signals[
                anchor_probability_column
            ].to_numpy(dtype=np.float64),
            "test_prediction": test_signals[
                anchor_prediction_column
            ].to_numpy(dtype=np.int64),
        }

    anchor = {
        "name": "anchor",
        "oof_probability": selection["candidates"][
            "anchor"
        ]["oof_probability"],
        "oof_prediction": selection["candidates"][
            "anchor"
        ]["oof_prediction"],
        "test_probability": test_signals[
            anchor_probability_column
        ].to_numpy(dtype=np.float64),
        "test_prediction": test_signals[
            anchor_prediction_column
        ].to_numpy(dtype=np.int64),
    }

    return anchor, forced, safe


context_anchor, context_forced, context_safe = extract_route_output(
    context_selection,
    context_test_signals,
    "context_lexical_prob",
    "context_anchor_pred",
)
null_anchor, null_forced, null_safe = extract_route_output(
    null_selection,
    null_test_signals,
    "null_step6_prob",
    "null_anchor_pred",
)


# =============================================================================
# COMBINE ROUTES IN ORIGINAL ROW ORDER
# =============================================================================

def combine_oof(
    context_probability,
    context_prediction,
    null_probability,
    null_prediction,
):
    output = official[
        ["id", "route", "fold", "label"]
    ].copy()
    output["probability_fair"] = np.nan
    output["prediction"] = -1

    context_mask = output["route"].eq("context")
    null_mask = output["route"].eq("null")

    output.loc[
        context_mask,
        "probability_fair",
    ] = context_probability
    output.loc[
        context_mask,
        "prediction",
    ] = context_prediction

    output.loc[
        null_mask,
        "probability_fair",
    ] = null_probability
    output.loc[
        null_mask,
        "prediction",
    ] = null_prediction

    output["prediction"] = output[
        "prediction"
    ].astype(int)

    return output


def make_submission(
    context_prediction,
    null_prediction,
):
    prediction_frame = competition_test[
        ["id", "route"]
    ].copy()
    prediction_frame["label"] = -1

    prediction_frame.loc[
        prediction_frame["route"].eq("context"),
        "label",
    ] = context_prediction

    prediction_frame.loc[
        prediction_frame["route"].eq("null"),
        "label",
    ] = null_prediction

    prediction_frame["label"] = prediction_frame[
        "label"
    ].astype(int)

    submission = prediction_frame[["id", "label"]].copy()

    if len(submission) != len(competition_test):
        raise ValueError("Submission row count mismatch.")
    if submission["id"].duplicated().any():
        raise ValueError("Duplicate submission IDs.")
    if submission["label"].isna().any():
        raise ValueError("Missing submission labels.")
    if not set(submission["label"].unique()).issubset({0, 1}):
        raise ValueError("Submission labels must be 0 or 1.")

    return submission


anchor_oof = combine_oof(
    context_anchor["oof_probability"],
    context_anchor["oof_prediction"],
    null_anchor["oof_probability"],
    null_anchor["oof_prediction"],
)
forced_oof = combine_oof(
    context_forced["oof_probability"],
    context_forced["oof_prediction"],
    null_forced["oof_probability"],
    null_forced["oof_prediction"],
)
safe_oof = combine_oof(
    context_safe["oof_probability"],
    context_safe["oof_prediction"],
    null_safe["oof_probability"],
    null_safe["oof_prediction"],
)

anchor_submission = make_submission(
    context_anchor["test_prediction"],
    null_anchor["test_prediction"],
)
forced_submission = make_submission(
    context_forced["test_prediction"],
    null_forced["test_prediction"],
)
safe_submission = make_submission(
    context_safe["test_prediction"],
    null_safe["test_prediction"],
)


# =============================================================================
# SAVE OUTPUTS
# =============================================================================

anchor_path = Path(
    "/kaggle/working/submission_hybrid_model_anchor_0745.csv"
)
forced_path = Path(
    "/kaggle/working/"
    "submission_hybrid_model_cv_fusion_forced.csv"
)
safe_path = Path(
    "/kaggle/working/"
    "submission_hybrid_model_cv_fusion.csv"
)

anchor_submission.to_csv(anchor_path, index=False)
forced_submission.to_csv(forced_path, index=False)
safe_submission.to_csv(safe_path, index=False)

anchor_oof.to_csv(
    FUSION_DIR / "anchor_oof.csv",
    index=False,
)
forced_oof.to_csv(
    FUSION_DIR / "forced_fusion_oof.csv",
    index=False,
)
safe_oof.to_csv(
    FUSION_DIR / "safe_fusion_oof.csv",
    index=False,
)

context_train_signals.to_csv(
    FUSION_DIR / "context_fusion_signals_oof.csv",
    index=False,
)
context_test_signals.to_csv(
    FUSION_DIR / "context_fusion_signals_test.csv",
    index=False,
)
null_train_signals.to_csv(
    FUSION_DIR / "null_fusion_signals_oof.csv",
    index=False,
)
null_test_signals.to_csv(
    FUSION_DIR / "null_fusion_signals_test.csv",
    index=False,
)

report = {
    "selection_metric": (
        "Primary: F1 on hallucinated label 0; "
        "secondary: Macro-F1, accuracy and label-1 F1"
    ),
    "validation": {
        "official_folds": 5,
        "fusion_inner_folds": FUSION_INNER_SPLITS,
        "method": (
            "outer-fold OOF validation with inner-fold "
            "model/weight and threshold selection"
        ),
        "test_inference": (
            "mean probability from five outer-fold stacking models"
        ),
    },
    "acceptance_rule": {
        "minimum_label0_gain": MIN_LABEL0_GAIN,
        "maximum_macro_drop": MAX_MACRO_DROP,
    },
    "context": {
        key: value
        for key, value in context_selection.items()
        if key != "candidates"
    },
    "null": {
        key: value
        for key, value in null_selection.items()
        if key != "candidates"
    },
    "overall": {
        "anchor_metrics": fusion_metrics(
            anchor_oof["label"],
            anchor_oof["prediction"],
        ),
        "forced_fusion_metrics": fusion_metrics(
            forced_oof["label"],
            forced_oof["prediction"],
        ),
        "safe_fusion_metrics": fusion_metrics(
            safe_oof["label"],
            safe_oof["prediction"],
        ),
    },
    "selected_methods": {
        "context_forced": context_forced["name"],
        "context_safe": context_safe["name"],
        "null_forced": null_forced["name"],
        "null_safe": null_safe["name"],
    },
    "submission_counts": {
        "anchor": {
            str(key): int(value)
            for key, value in anchor_submission[
                "label"
            ].value_counts().to_dict().items()
        },
        "forced": {
            str(key): int(value)
            for key, value in forced_submission[
                "label"
            ].value_counts().to_dict().items()
        },
        "safe": {
            str(key): int(value)
            for key, value in safe_submission[
                "label"
            ].value_counts().to_dict().items()
        },
    },
    "files": {
        "anchor_submission": str(anchor_path),
        "forced_fusion_submission": str(forced_path),
        "safe_fusion_submission": str(safe_path),
        "fusion_directory": str(FUSION_DIR),
    },
}

with open(
    FUSION_DIR / "cv_fusion_report.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        report,
        file,
        ensure_ascii=False,
        indent=2,
    )

print("=" * 96)
print("FULL CROSS-VALIDATED FUSION RESULT")
print("=" * 96)
print(json.dumps(report, ensure_ascii=False, indent=2))

print("\nFinal files:")
print("Anchor:", anchor_path)
print("Forced fusion:", forced_path)
print("Safe fusion:", safe_path)

display(safe_submission.head())


## Final output files

- Original proven anchor: `/kaggle/working/submission_hybrid_model_anchor_0745.csv`
- Best forced CV fusion: `/kaggle/working/submission_hybrid_model_cv_fusion_forced.csv`
- Recommended safe output: `/kaggle/working/submission_hybrid_model_cv_fusion.csv`
- Full validation report: `/kaggle/working/hybrid_cv_fusion/cv_fusion_report.json`

Use the safe file for the final project unless the forced fusion shows a clear, consistent OOF improvement and the route-level audit supports it.
